# Moirai + Fourier + semantic neighbours + ensemble (h = 6)

Upstream source: `Morai_sem_improv_collab.ipynb` in the thesis workbench.

**Input**: `KEYWORDS_DIR_20 / *.parquet` (semantic 20-neighbour panels).
**Output**: `moirai_results_dir('sem20')` and `moirai_results_dir('dtw20_fourier_small_h6')` / `moirai_results_dir('dtw20_fourier_small_h6_final')` / `moirai_results_dir('dtw20_fourier_small_h6_ensemble')` depending on which iteration cell is executed.

The notebook walks through v1 -> v4.5 of the combined Fourier + semantic-neighbour + ensemble design; each cell is a progressive refinement over the previous one. The paper reports the v4.5 ('PROD') configuration.


## Canonical cell for the paper

The paper reports the **semantic-based Moirai** row from the cell titled *Moirai H=6 - PROD v4.5 (pure Moirai + k=9 calendar + α=0.25 anchor + 3 %% floor + dtype/shape safety)*. The cell consumes `KEYWORDS_DIR_20` (top-20 semantic-neighbour panels) and combines pretrained Moirai with Fourier calendar features, recency anchoring and a 3 %% floor. Predecessor cells (v1 → v4.4) are the progressive methodological iterations and are retained to make the evolution auditable.


In [ ]:
# --- Paper-repo bootstrap (replaces original Colab `drive.mount` cell) ---
import sys, os
from pathlib import Path

_REPO_SHARED = Path('..', '_shared').resolve()
if str(_REPO_SHARED) not in sys.path:
    sys.path.insert(0, str(_REPO_SHARED))

from paths import (  # noqa: E402
    ensure_env, DATA_ROOT, DTW_DFS_DIR, KEYWORDS_DIR_20, moirai_results_dir,
)
from api_keys import get_hf_token  # noqa: E402

ensure_env()
_hf_token = get_hf_token()
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token


In [ ]:
# --- Pin a stable stack for Py3.11 + uni2ts forecast path ---
!pip -q uninstall -y numpy pandas scipy transformers tokenizers huggingface-hub lightning torchmetrics pillow

# Core ABI pair (NumPy 1.26 + SciPy 1.11) + pandas
!pip -q install --no-deps numpy==1.26.4 pandas==2.2.2 scipy==1.11.4 pillow==11.0.0

# HF/Lightning stack that previously clashed
!pip -q install --no-deps transformers==4.46.2 tokenizers==0.22.0 huggingface-hub==0.36.0 lightning==2.3.3 torchmetrics==1.3.2

# (optional) make sure these are present
!pip -q install uni2ts==2.0.0 gluonts==0.14.3 polars==1.32.1 tqdm==4.66.5 pyarrow==22.0.0

# ---- Force runtime restart so NumPy/Pandas C-extensions reload cleanly ----
import os, signal, sys
os.kill(os.getpid(), 9)


In [ ]:
# Fix the HF mismatch: keep your current transformers, downgrade tokenizers
!pip -q uninstall -y tokenizers
!pip -q install --no-deps tokenizers==0.20.3

# sanity check
import tokenizers, transformers
print("tokenizers:", tokenizers.__version__)
print("transformers:", transformers.__version__)


In [ ]:
# ==== Moirai H=6 (auto-scales for 110–127 weeks) ====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch


# ------------ Paths ------------
IN_DIR  = KEYWORDS_DIR_20 / "rent.parquet"
df=pd.read_parquet(IN_DIR)
print(df.columns)


In [ ]:
# ==== Moirai H=6 (auto-scales for 110–127 weeks) ====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ------------ Fixed eval params ------------
FREQ        = "W"
H           = 6
NUM_SAMPLES = 200        # 120–300: higher = smoother median, slower
BATCH_SIZE  = 64
CTX_CAP     = 96         # cap context for speed/stability
CTX_FLOOR   =  FortyEight = 48

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device)

# ------------ Paths ------------
IN_DIR  = KEYWORDS_DIR_20
OUT_DIR = moirai_results_dir("sem20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_H6_auto_adaptive.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# ------------ IO + helpers ------------
EXCLUDE_COLS = {"week","cpc_week","keyword","adcost_sum","adclicks_sum","year","week_num"}

def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns: return None
    base = (
        df_pl.select(pl.col("week"), pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week").sort("ds").filter(pl.col("y").is_not_null())
    ).to_pandas()
    base["ds"] = pd.to_datetime(base["ds"])
    base["y"]  = pd.to_numeric(base["y"], errors="coerce").astype(np.float32)
    base = base.sort_values("ds").reset_index(drop=True)
    return base if len(base) >= (H+32) else None

def add_calendar(pdf: pd.DataFrame, k=3) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    return pdf

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred))/2.0
    m = denom != 0
    return float(100.0*np.mean(np.abs(y_pred[m]-y_true[m])/denom[m]))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((y_pred-y_true)**2)))

# ---- Model (load once)
try:
    module
except NameError:
    module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

def make_auto_feature_plan(n_total: int, h: int):
    """Return dict with adaptive windows given total length."""
    n_train = n_total - h
    # lag budget ≈ 20% of train (min 8, max 26)
    max_lag = int(max(8, min(26, 0.2*n_train)))
    # rolling windows: subset of {6,12,26,52} that fit history
    roll_candidates = [6,12,26,52]
    roll_means = [w for w in roll_candidates if w <= max(max_lag, 12)]
    roll_stds  = [w for w in [6,12,26] if w <= max(max_lag, 12)]
    # EMAs: safe short spans
    ema_spans = [s for s in [6,12] if s <= max_lag]
    # warmup: ensure enough to compute features (cap to 1/3 of train)
    warm = max(max_lag, max(roll_means+[6]), max(roll_stds+[6]), max(ema_spans+[6]))
    warm = int(min(warm, max(12, n_train//3)))
    # context: use ~70% of post-warm train, bounded
    ctx = n_train - warm
    ctx = max(CTX_FLOOR, min(CTX_CAP, int(0.7*ctx)))
    return {
        "lags": list(range(1, max_lag+1)),
        "roll_means": roll_means,
        "roll_stds": roll_stds,
        "ema_spans": ema_spans,
        "warm": warm,
        "ctx": ctx,
        "neigh_lags": list(range(1, min(6, max_lag)+1))
    }

def add_past_feats(pdf: pd.DataFrame, target: str, plan: dict) -> pd.DataFrame:
    pdf = pdf.copy()
    for L in plan["lags"]:
        pdf[f"{target}_lag_{L}"] = pdf[target].shift(L)
    for W in plan["roll_means"]:
        pdf[f"{target}_rollmean_{W}"] = pdf[target].rolling(W, min_periods=1).mean().shift(1)
    for W in plan["roll_stds"]:
        pdf[f"{target}_rollstd_{W}"] = pdf[target].rolling(W, min_periods=2).std().shift(1)
    for S in plan["ema_spans"]:
        pdf[f"{target}_ema_{S}"] = pdf[target].ewm(span=S, min_periods=1, adjust=False).mean().shift(1)
    return pdf

def pick_top_neighbors(tr: pd.DataFrame, k=3):
    # Use available cpc_sim_* columns (if any), pick by corr on TRAIN ONLY
    cand = [c for c in tr.columns if c.startswith("cpc_lastweek_") and "lastweek" not in c]
    if not cand: return []
    y = tr["y"]
    cors=[]
    for c in cand:
        s = pd.to_numeric(tr[c], errors="coerce")
        if s.isna().all(): continue
        r = y.corr(s)
        if pd.isna(r): continue
        cors.append((abs(r), c))
    cors.sort(reverse=True)
    return [c for _,c in cors[:k]]

def run_one(pdf: pd.DataFrame):
    # Calendar (future-known)
    pdf = add_calendar(pdf, k=3)

    # Log1p transform target for stability; keep original y for eval
    pdf = pdf.copy()
    pdf["y_orig"] = pdf["y"].astype(np.float32)
    pdf["y"] = np.log1p(np.clip(pdf["y_orig"], a_min=0, a_max=None)).astype(np.float32)

    tr, te = split_train_test(pdf, H)
    if tr is None: return np.nan, np.nan, 0, 0

    plan = make_auto_feature_plan(len(pdf), H)

    # Build past features on y (log-space)
    tr_p = add_past_feats(tr, "y", plan)
    te_p = add_past_feats(pd.concat([tr.tail(plan["warm"]), te], ignore_index=True), "y", plan).iloc[plan["warm"]:]

    # Neighbor past lags (if sim columns exist) — no leakage
    neigh = pick_top_neighbors(tr, k=3)
    for c in neigh:
        for L in plan["neigh_lags"]:
            tr_p[f"{c}_lag{L}"] = tr[c].shift(L)
            te_concat = pd.concat([tr[[c]].tail(L), te[[c]]], ignore_index=True)
            te_p[f"{c}_lag{L}"] = te_concat[c].shift(L).iloc[L:].values

    past_cols = [c for c in tr_p.columns if c.startswith(("y_lag_","y_rollmean_","y_rollstd_","y_ema_"))] + \
                [f"{c}_lag{L}" for c in neigh for L in plan["neigh_lags"]]

    # Drop warmup
    if len(tr_p) <= plan["warm"] + 8:
        return np.nan, np.nan, 0, 0
    tr_base = tr.iloc[plan["warm"]:].copy()
    tr_p    = tr_p.iloc[plan["warm"]:].copy()
    te_base = te.copy()

    # Future-known covars: calendar only (safe)
    fut_cols = [c for c in pdf.columns if c.startswith("cal_")]

    entry = {
        "start": pd.Timestamp(tr_base["ds"].iloc[0]),
        "target": tr_base["y"].to_numpy(np.float32),          # log-space
    }
    if fut_cols:
        Xp = np.stack([tr_base[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        Xf = np.stack([te_base[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)
    if past_cols:
        entry["past_feat_dynamic_real"] = np.stack([tr_p[c].to_numpy(np.float32) for c in past_cols], axis=0)

    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=len(fut_cols),
        past_feat_dynamic_real_dim=len(past_cols),
        context_length=plan["ctx"],
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    fc = next(predictor.predict(ListDataset([entry], freq=FREQ)))

    # Median over samples in log-space → inverse-transform to original space
    pred_log = np.asarray(np.median(fc.samples, axis=0), dtype=np.float32) if fc.samples is not None else np.asarray(fc.mean, dtype=np.float32)
    yhat = np.expm1(pred_log)                                  # back to original scale
    y    = te_base["y_orig"].to_numpy(np.float32)              # original target for eval
    return smape(y, yhat), rmse(y, yhat), len(fut_cols), len(past_cols)

# ------------ Run (H=6 only) ------------
rows=[]
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} files.")
for p in tqdm(files, desc="H=6 auto-adaptive"):
    pdf = prep_keyword_pdf(p)
    if pdf is None:
        continue
    try:
        s, r, kf, kp = run_one(pdf)
    except Exception as e:
        print(f"[WARN] {p.stem} failed: {e}")
        s, r, kf, kp = np.nan, np.nan, 0, 0
    rows.append({
        "keyword": p.stem, "exog_setting": "auto+past_only+calendar",
        "horizon_weeks": H, "smape_canonical": s, "rmse": r,
        "n_future_covars": kf, "n_past_covars": kp, "n_total": len(pdf)
    })

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print("✅ Done. Metrics:", OUT_CSV)

valid = out.dropna(subset=["smape_canonical","rmse"])
if not valid.empty:
    g = valid.agg(
        count=("rmse","count"),
        sMAPE_mean=("smape_canonical","mean"),
        sMAPE_median=("smape_canonical","median"),
        RMSE_mean=("rmse","mean"),
        RMSE_median=("rmse","median"),
    )
    display(g)
    # quick percent-below thresholds
    for t in [10,15,20,25,30,40]:
        share = (valid["smape_canonical"] <= t).mean()
        print(f"sMAPE≤{t:>2}: {share:.3f}")
else:
    print("No valid rows.")


In [ ]:
# ==== Moirai H=6 (auto-scales for 110–127 weeks) – IMPROVED ====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ------------ Fixed eval params ------------
FREQ        = "W"
H           = 6
NUM_SAMPLES = 256       # smoother median (200 -> 256)
BATCH_SIZE  = 64
CTX_CAP     = 128       # longer context (96 -> 128)
CTX_FLOOR   = 48        # fixed invalid assignment

# Neighbor column prefix (matches your parquet schema)
NEIGH_PREFIX = "cpc_lastweek__"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device)

# ------------ Paths ------------
IN_DIR  = KEYWORDS_DIR_20
OUT_DIR = moirai_results_dir("sem20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_H6_auto_adaptive.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# ------------ IO + helpers ------------
EXCLUDE_COLS = {"week","cpc_week","keyword","adcost_sum","adclicks_sum","year","week_num"}

def iso_week_to_date(wwyyyy: str) -> date:
    # expects "WW-YYYY" (e.g., "52-2024")
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns: return None
    base = (
        df_pl.select(pl.col("week"), pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week").sort("ds").filter(pl.col("y").is_not_null())
    ).to_pandas()
    base["ds"] = pd.to_datetime(base["ds"])
    base["y"]  = pd.to_numeric(base["y"], errors="coerce").astype(np.float32)
    base = base.sort_values("ds").reset_index(drop=True)
    return base if len(base) >= (H+32) else None

def add_calendar(pdf: pd.DataFrame, k=5) -> pd.DataFrame:
    """Add richer annual Fourier terms + centered trend."""
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    return pdf

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred))/2.0
    m = denom != 0
    return float(100.0*np.mean(np.abs(y_pred[m]-y_true[m])/denom[m]))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((y_pred-y_true)**2)))

# ---- Model (load once)
try:
    module
except NameError:
    module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

def make_auto_feature_plan(n_total: int, h: int):
    """Return dict with adaptive windows given total length (slightly longer lags)."""
    n_train = n_total - h
    # lag budget ≈ 25% of train (min 8, max 52) to let yearly signals in longer series
    max_lag = int(max(8, min(52, 0.25*n_train)))
    # rolling windows: subset of {6,12,26,52} that fit history
    roll_candidates = [6,12,26,52]
    roll_means = [w for w in roll_candidates if w <= max(max_lag, 12)]
    roll_stds  = [w for w in [6,12,26] if w <= max(max_lag, 12)]
    # EMAs: safe short spans
    ema_spans = [s for s in [6,12] if s <= max_lag]
    # warmup: ensure enough to compute features (cap to 1/3 of train)
    warm = max(max_lag, max(roll_means+[6]), max(roll_stds+[6]), max(ema_spans+[6]))
    warm = int(min(warm, max(12, n_train//3)))
    # context: use ~70% of post-warm train, bounded
    ctx = n_train - warm
    ctx = max(CTX_FLOOR, min(CTX_CAP, int(0.7*ctx)))
    return {
        "lags": list(range(1, max_lag+1)),
        "roll_means": roll_means,
        "roll_stds": roll_stds,
        "ema_spans": ema_spans,
        "warm": warm,
        "ctx": ctx,
        "neigh_lags": list(range(1, min(6, max_lag)+1))
    }

def add_past_feats(pdf: pd.DataFrame, target: str, plan: dict) -> pd.DataFrame:
    pdf = pdf.copy()
    for L in plan["lags"]:
        pdf[f"{target}_lag_{L}"] = pdf[target].shift(L)
    for W in plan["roll_means"]:
        pdf[f"{target}_rollmean_{W}"] = pdf[target].rolling(W, min_periods=1).mean().shift(1)
    for W in plan["roll_stds"]:
        pdf[f"{target}_rollstd_{W}"] = pdf[target].rolling(W, min_periods=2).std().shift(1)
    for S in plan["ema_spans"]:
        pdf[f"{target}_ema_{S}"] = pdf[target].ewm(span=S, min_periods=1, adjust=False).mean().shift(1)
    return pdf

def pick_top_neighbors(tr: pd.DataFrame, k=3):
    """Pick up to k neighbor columns by absolute Pearson correlation on TRAIN ONLY."""
    cand = [c for c in tr.columns if c.startswith(NEIGH_PREFIX)]
    if not cand:
        return []
    y = pd.to_numeric(tr["y"], errors="coerce")
    cors = []
    for c in cand:
        s = pd.to_numeric(tr[c], errors="coerce")
        if s.isna().all():
            continue
        r = y.corr(s)  # Pearson; switch to method="spearman" if you expect monotonic non-linear
        if pd.isna(r):
            continue
        cors.append((abs(r), c))
    cors.sort(reverse=True)
    return [c for _, c in cors[:k]]

def run_one(pdf: pd.DataFrame):
    # Calendar (future-known)
    pdf = add_calendar(pdf, k=5)

    # Keep original target for eval
    pdf = pdf.copy()
    pdf["y_orig"] = pdf["y"].astype(np.float32)
    # Log1p transform for stability (still in original scale here)
    pdf["y"] = np.log1p(np.clip(pdf["y_orig"], a_min=0, a_max=None)).astype(np.float32)

    tr, te = split_train_test(pdf, H)
    if tr is None:
        return np.nan, np.nan, 0, 0

    plan = make_auto_feature_plan(len(pdf), H)

    # --- Standardize log target on TRAIN ONLY; apply same params to TE ---
    mean_y = tr["y"].mean()
    std_y  = tr["y"].std() + 1e-8
    tr = tr.copy()
    te = te.copy()
    tr["y"] = (tr["y"] - mean_y) / std_y
    te["y"] = (te["y"] - mean_y) / std_y

    # Build past features on standardized log-y
    tr_p = add_past_feats(tr, "y", plan)

    # For TE past feats, concatenate last 'warm' rows from TRAIN with TE and drop warmup
    te_build = pd.concat([tr.tail(plan["warm"]), te], ignore_index=True)
    te_p_all = add_past_feats(te_build, "y", plan)
    te_p     = te_p_all.iloc[plan["warm"]:]  # align to TE rows

    # Neighbor past lags (if semantic-sim columns exist) — no leakage
    neigh = pick_top_neighbors(tr, k=3)
    for c in neigh:
        for L in plan["neigh_lags"]:
            # TRAIN: simple shift(L)
            tr_p[f"{c}_lag{L}"] = tr[c].shift(L)

            # TEST: use last L rows from TRAIN + TE, then shift by 1 (strictly previous)
            te_concat = pd.concat([tr[[c]].iloc[-L:], te[[c]]], ignore_index=True)
            te_shift1 = te_concat[c].shift(1)
            # take the last len(te) values after shift -> aligns step t with t-1 value
            te_vals = te_shift1.iloc[-len(te):].values
            te_p[f"{c}_lag{L}"] = te_vals

    past_cols = [c for c in tr_p.columns if c.startswith(("y_lag_","y_rollmean_","y_rollstd_","y_ema_"))] + \
                [f"{c}_lag{L}" for c in neigh for L in plan["neigh_lags"]]

    # Drop warmup portion from TRAIN past features and align bases
    if len(tr_p) <= plan["warm"] + 8:
        return np.nan, np.nan, 0, 0

    tr_base = tr.iloc[plan["warm"]:].copy()
    tr_p    = tr_p.iloc[plan["warm"]:].copy()
    te_base = te.copy()

    # Future-known covars: calendar only (safe)
    fut_cols = [c for c in pdf.columns if c.startswith("cal_")]

    entry = {
        "start": pd.Timestamp(tr_base["ds"].iloc[0]),
        "target": tr_base["y"].to_numpy(np.float32),          # standardized log-space
    }
    if fut_cols:
        Xp = np.stack([tr_base[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        Xf = np.stack([te_base[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)
    if past_cols:
        entry["past_feat_dynamic_real"] = np.stack([tr_p[c].to_numpy(np.float32) for c in past_cols], axis=0)

    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=len(fut_cols),
        past_feat_dynamic_real_dim=len(past_cols),
        context_length=plan["ctx"],
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    fc = next(predictor.predict(ListDataset([entry], freq=FREQ)))

    # Median over samples in standardized log-space -> unstandardize -> inverse log1p
    pred_stdlog = np.asarray(np.median(fc.samples, axis=0), dtype=np.float32) if fc.samples is not None else np.asarray(fc.mean, dtype=np.float32)
    pred_log    = pred_stdlog * std_y + mean_y
    yhat        = np.expm1(pred_log)                                  # back to original space
    y_true      = te_base["y_orig"].to_numpy(np.float32)              # original target for eval

    return smape(y_true, yhat), rmse(y_true, yhat), len(fut_cols), len(past_cols)

# ------------ Run (H=6 only) ------------
rows=[]
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} files.")
for p in tqdm(files, desc="H=6 auto-adaptive"):
    pdf = prep_keyword_pdf(p)
    if pdf is None:
        continue
    try:
        s, r, kf, kp = run_one(pdf)
    except Exception as e:
        print(f"[WARN] {p.stem} failed: {e}")
        s, r, kf, kp = np.nan, np.nan, 0, 0
    rows.append({
        "keyword": p.stem, "exog_setting": "auto+past_only+calendar+neighbors",
        "horizon_weeks": H, "smape_canonical": s, "rmse": r,
        "n_future_covars": kf, "n_past_covars": kp, "n_total": len(pdf)
    })

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print("✅ Done. Metrics:", OUT_CSV)

valid = out.dropna(subset=["smape_canonical","rmse"])
if not valid.empty:
    g = valid.agg(
        count=("rmse","count"),
        sMAPE_mean=("smape_canonical","mean"),
        sMAPE_median=("smape_canonical","median"),
        RMSE_mean=("rmse","mean"),
        RMSE_median=("rmse","median"),
    )
    display(g)
    # quick percent-below thresholds
    for t in [10,15,20,25,30,40]:
        share = (valid["smape_canonical"] <= t).mean()
        print(f"sMAPE≤{t:>2}: {share:.3f}")
else:
    print("No valid rows.")


In [ ]:
# ==== Moirai H=6 (auto-scales for 110–127 weeks) – IMPROVED v2 (neighbors+past exog lags+winsor) ====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ------------ Fixed eval params ------------
FREQ        = "W"
H           = 6
NUM_SAMPLES = 256        # smoother median
BATCH_SIZE  = 64
CTX_CAP     = 128        # longer context
CTX_FLOOR   = 48
WINSOR_P    = 0.995      # cap extreme CPC before log1p
NEIGH_PREFIX = "cpc_lastweek__"
NEIGH_K     = 5          # try up to 5 neighbors
NEIGH_METHOD = "pearson" # or "spearman"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device)

# ------------ Paths ------------
IN_DIR  = KEYWORDS_DIR_20
OUT_DIR = moirai_results_dir("sem20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_H6_auto_adaptive.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# ------------ IO + helpers ------------
EXCLUDE_COLS = {"week","cpc_week","keyword","adcost_sum","adclicks_sum","year","week_num"}

def iso_week_to_date(wwyyyy: str) -> date:
    # expects "WW-YYYY" (e.g., "52-2024")
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns: return None
    base = (
        df_pl.select(pl.col("week"), pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week").sort("ds").filter(pl.col("y").is_not_null())
    ).to_pandas()
    base["ds"] = pd.to_datetime(base["ds"])
    base["y"]  = pd.to_numeric(base["y"], errors="coerce").astype(np.float32)
    base = base.sort_values("ds").reset_index(drop=True)
    return base if len(base) >= (H+32) else None

def add_calendar(pdf: pd.DataFrame, k=5) -> pd.DataFrame:
    """Add richer annual Fourier terms + centered trend."""
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    return pdf

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred))/2.0
    m = denom != 0
    return float(100.0*np.mean(np.abs(y_pred[m]-y_true[m])/denom[m]))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((y_pred-y_true)**2)))

# ---- Model (load once)
try:
    module
except NameError:
    module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

def make_auto_feature_plan(n_total: int, h: int):
    """Return dict with adaptive windows given total length (slightly longer lags)."""
    n_train = n_total - h
    # lag budget ≈ 25% of train (min 8, max 52) to let yearly signals in longer series
    max_lag = int(max(8, min(52, 0.25*n_train)))
    # rolling windows: subset of {6,12,26,52} that fit history
    roll_candidates = [6,12,26,52]
    roll_means = [w for w in roll_candidates if w <= max(max_lag, 12)]
    roll_stds  = [w for w in [6,12,26] if w <= max(max_lag, 12)]
    # EMAs: safe short spans
    ema_spans = [s for s in [6,12] if s <= max_lag]
    # warmup: ensure enough to compute features (cap to 1/3 of train)
    warm = max(max_lag, max(roll_means+[6]), max(roll_stds+[6]), max(ema_spans+[6]))
    warm = int(min(warm, max(12, n_train//3)))
    # context: use ~70% of post-warm train, bounded
    ctx = n_train - warm
    ctx = max(CTX_FLOOR, min(CTX_CAP, int(0.7*ctx)))
    return {
        "lags": list(range(1, max_lag+1)),
        "roll_means": roll_means,
        "roll_stds": roll_stds,
        "ema_spans": ema_spans,
        "warm": warm,
        "ctx": ctx,
        "neigh_lags": list(range(1, min(6, max_lag)+1)),
        "exog_lags": list(range(1, min(6, max_lag)+1))
    }

def add_past_feats(pdf: pd.DataFrame, target: str, plan: dict) -> pd.DataFrame:
    pdf = pdf.copy()
    for L in plan["lags"]:
        pdf[f"{target}_lag_{L}"] = pdf[target].shift(L)
    for W in plan["roll_means"]:
        pdf[f"{target}_rollmean_{W}"] = pdf[target].rolling(W, min_periods=1).mean().shift(1)
    for W in plan["roll_stds"]:
        pdf[f"{target}_rollstd_{W}"] = pdf[target].rolling(W, min_periods=2).std().shift(1)
    for S in plan["ema_spans"]:
        pdf[f"{target}_ema_{S}"] = pdf[target].ewm(span=S, min_periods=1, adjust=False).mean().shift(1)
    return pdf

def pick_top_neighbors(tr: pd.DataFrame, k=NEIGH_K, method=NEIGH_METHOD):
    """Pick up to k neighbor columns by absolute correlation on TRAIN ONLY, with simple de-dup."""
    cand = [c for c in tr.columns if c.startswith(NEIGH_PREFIX)]
    if not cand:
        return []
    y = pd.to_numeric(tr["y"], errors="coerce")
    scores = []
    for c in cand:
        s = pd.to_numeric(tr[c], errors="coerce")
        if s.isna().all():
            continue
        r = y.corr(s, method=method)
        if pd.isna(r):
            continue
        scores.append((abs(r), c))
    scores.sort(reverse=True)
    # simple diversity: avoid picking nearly-duplicate neighbors
    selected = []
    selected_series = []
    for _, c in scores:
        s = pd.to_numeric(tr[c], errors="coerce")
        if not selected_series:
            selected.append(c); selected_series.append(s);
        else:
            ok = True
            for s2 in selected_series:
                if abs(pd.Series(s).corr(pd.Series(s2))) > 0.95:
                    ok = False; break
            if ok:
                selected.append(c); selected_series.append(s)
        if len(selected) >= k:
            break
    return selected

def list_past_exog_cols(df: pd.DataFrame):
    """
    Choose safe exogenous drivers to use ONLY as past features (no future leakage).
    - Explicit last-week fields are already safe: *_last_week
    - Counts/splits that are contemporaneous -> we lag them.
    """
    exog = []
    for c in df.columns:
        if c in ("ds","y","y_orig"):
            continue
        if c.startswith("cal_"):  # calendar is future-known; handled separately
            continue
        if c.startswith(("cpc_lastweek__")):
            continue  # handled in neighbor block
        if c in ("avg_sim_top25_last_week","n_sim_last_week"):
            exog.append(c)
        if c.startswith(("n_dev_","n_st_","impressions_sum","adclicks_sum","adcost_sum")):
            exog.append(c)
    return list(dict.fromkeys(exog))  # unique & preserve order

def build_past_exog(tr: pd.DataFrame, te: pd.DataFrame, exog_cols: list, plan: dict):
    """Return (tr_ex, te_ex, names) with lagged past exogenous features (no leakage)."""
    tr_ex = pd.DataFrame(index=tr.index)
    te_ex = pd.DataFrame(index=te.index)
    names = []
    if not exog_cols:
        return tr_ex, te_ex, names
    for c in exog_cols:
        # TE builder uses last max(L) rows from TRAIN + TE
        Lmax = max(plan["exog_lags"])
        te_concat = pd.concat([tr[[c]].iloc[-Lmax:], te[[c]]], ignore_index=True)
        for L in plan["exog_lags"]:
            cname = f"{c}_lag{L}"
            tr_ex[cname] = tr[c].shift(L)
            te_ex[cname] = te_concat[c].shift(1).iloc[-len(te):].values  # strictly previous
            names.append(cname)
    return tr_ex, te_ex, names

def run_one(pdf: pd.DataFrame):
    # Calendar (future-known)
    pdf = add_calendar(pdf, k=5)

    # Winsorize extreme CPCs, keep original target for eval
    pdf = pdf.copy()
    pdf["y_orig"] = pdf["y"].astype(np.float32)
    cap = np.nanquantile(pdf["y_orig"].values, WINSOR_P) if np.isfinite(pdf["y_orig"]).any() else None
    if cap is not None and np.isfinite(cap):
        pdf["y_cap"] = np.clip(pdf["y_orig"], 0, cap).astype(np.float32)
    else:
        pdf["y_cap"] = np.clip(pdf["y_orig"], 0, None).astype(np.float32)

    # Log1p transform for stability
    pdf["y"] = np.log1p(pdf["y_cap"]).astype(np.float32)

    tr, te = split_train_test(pdf, H)
    if tr is None:
        return np.nan, np.nan, 0, 0

    plan = make_auto_feature_plan(len(pdf), H)

    # --- Standardize log target on TRAIN ONLY; apply same params to TE ---
    mean_y = tr["y"].mean()
    std_y  = tr["y"].std() + 1e-8
    tr = tr.copy()
    te = te.copy()
    tr["y"] = (tr["y"] - mean_y) / std_y
    te["y"] = (te["y"] - mean_y) / std_y

    # Build past features on standardized log-y
    tr_p = add_past_feats(tr, "y", plan)

    # For TE past feats, concatenate last 'warm' rows from TRAIN with TE and drop warmup
    te_build = pd.concat([tr.tail(plan["warm"]), te], ignore_index=True)
    te_p_all = add_past_feats(te_build, "y", plan)
    te_p     = te_p_all.iloc[plan["warm"]:]  # align to TE rows

    # Neighbor past lags (semantic peers) — no leakage
    neigh = pick_top_neighbors(tr, k=NEIGH_K, method=NEIGH_METHOD)
    for c in neigh:
        for L in plan["neigh_lags"]:
            tr_p[f"{c}_lag{L}"] = tr[c].shift(L)
            te_concat = pd.concat([tr[[c]].iloc[-L:], te[[c]]], ignore_index=True)
            te_shift1 = te_concat[c].shift(1)
            te_vals = te_shift1.iloc[-len(te):].values
            te_p[f"{c}_lag{L}"] = te_vals

    # Past exogenous (counts, last-week similarity, devices, spend, etc.) — lagged only
    exog_cols = list_past_exog_cols(pdf)
    tr_ex, te_ex, ex_names = build_past_exog(tr, te, exog_cols, plan)
    # Merge into past features frames
    if not tr_ex.empty:
        tr_p = pd.concat([tr_p, tr_ex], axis=1)
        te_p = pd.concat([te_p, te_ex], axis=1)

    past_cols = [c for c in tr_p.columns if c.startswith(("y_lag_","y_rollmean_","y_rollstd_","y_ema_"))] + \
                [c for c in tr_p.columns if c.endswith(tuple([f"_lag{L}" for L in plan["neigh_lags"]]))]

    # Also include the explicit exog lag names
    past_cols = list(dict.fromkeys(past_cols + ex_names))

    # Drop warmup portion from TRAIN past features and align bases
    if len(tr_p) <= plan["warm"] + 8:
        return np.nan, np.nan, 0, 0

    tr_base = tr.iloc[plan["warm"]:].copy()
    tr_p    = tr_p.iloc[plan["warm"]:].copy()
    te_base = te.copy()
    # --- after building tr_p / te_p (just before "if len(tr_p) <= plan['warm'] + 8:")
    # Keep only numeric columns and coerce to float32
    numeric_cols_tr = [c for c in tr_p.columns if pd.api.types.is_numeric_dtype(tr_p[c])]
    numeric_cols_te = [c for c in te_p.columns if pd.api.types.is_numeric_dtype(te_p[c])]
    tr_p = tr_p[numeric_cols_tr].apply(pd.to_numeric, errors="coerce").astype(np.float32)
    te_p = te_p[numeric_cols_te].apply(pd.to_numeric, errors="coerce").astype(np.float32)

    # Future-known covars: calendar only (safe)
    fut_cols = [c for c in pdf.columns if c.startswith("cal_")]

    entry = {
        "start": pd.Timestamp(tr_base["ds"].iloc[0]),
        "target": tr_base["y"].to_numpy(np.float32),          # standardized log-space
    }
    if fut_cols:
        Xp = np.stack([tr_base[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        Xf = np.stack([te_base[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)
    if past_cols:
        entry["past_feat_dynamic_real"] = np.stack([tr_p[c].to_numpy(np.float32) for c in past_cols], axis=0)

    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=len(fut_cols),
        past_feat_dynamic_real_dim=len(past_cols),
        context_length=plan["ctx"],
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    fc = next(predictor.predict(ListDataset([entry], freq=FREQ)))

    # Median over samples in standardized log-space -> unstandardize -> inverse log1p
    pred_stdlog = np.asarray(np.median(fc.samples, axis=0), dtype=np.float32) if fc.samples is not None else np.asarray(fc.mean, dtype=np.float32)
    pred_log    = pred_stdlog * std_y + mean_y
    yhat        = np.expm1(pred_log)                                  # back to original space
    y_true      = te_base["y_cap"].to_numpy(np.float32)               # evaluate vs capped original

    return smape(y_true, yhat), rmse(y_true, yhat), len(fut_cols), len(past_cols)

# ------------ Run (H=6 only) ------------
rows=[]
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} files.")
for p in tqdm(files, desc="H=6 auto-adaptive"):
    pdf = prep_keyword_pdf(p)
    if pdf is None:
        continue
    try:
        s, r, kf, kp = run_one(pdf)
    except Exception as e:
        print(f"[WARN] {p.stem} failed: {e}")
        s, r, kf, kp = np.nan, np.nan, 0, 0
    rows.append({
        "keyword": p.stem, "exog_setting": "auto+past_only+calendar+neighbors+exoglags",
        "horizon_weeks": H, "smape_canonical": s, "rmse": r,
        "n_future_covars": kf, "n_past_covars": kp, "n_total": len(pdf)
    })

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print("✅ Done. Metrics:", OUT_CSV)

valid = out.dropna(subset=["smape_canonical","rmse"])
if not valid.empty:
    g = valid.agg(
        count=("rmse","count"),
        sMAPE_mean=("smape_canonical","mean"),
        sMAPE_median=("smape_canonical","median"),
        RMSE_mean=("rmse","mean"),
        RMSE_median=("rmse","median"),
    )
    display(g)
    # quick percent-below thresholds
    for t in [10,15,20,25,30,40]:
        share = (valid["smape_canonical"] <= t).mean()
        print(f"sMAPE≤{t:>2}: {share:.3f}")
else:
    print("No valid rows.")


In [ ]:
# ==== Moirai H=6 – IMPROVED v4.2 (hard feature alignment + dtype safety + optional pure Moirai) ====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ------------ Fixed eval params ------------
FREQ        = "W"
H           = 6
NUM_SAMPLES = 256
BATCH_SIZE  = 64
CTX_CAP     = 128
CTX_FLOOR   = 48
WINSOR_P    = 0.995

NEIGH_PREFIX = "cpc_lastweek__"
NEIGH_K      = 6
NEIGH_ROLL_W = 52   # recency window for neighbor selection (spearman)

# Blend settings
PURE_MOIRAI = True         # set True for Moirai-only outputs (no ridge/seasonal)
W_MOIRAI   = 0.65
W_RIDGE    = 0.25
W_SEASONAL = 0.10

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device)

# ------------ Paths ------------
IN_DIR  = KEYWORDS_DIR_20
OUT_DIR = moirai_results_dir("sem20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_H6_auto_adaptive.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# ------------ IO + helpers ------------
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")  # expects "WW-YYYY"
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns: return None
    base = (
        df_pl.select(pl.col("week"), pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week").sort("ds").filter(pl.col("y").is_not_null())
    ).to_pandas()
    base["ds"] = pd.to_datetime(base["ds"])
    base["y"]  = pd.to_numeric(base["y"], errors="coerce").astype(np.float32)
    base = base.sort_values("ds").reset_index(drop=True)
    return base if len(base) >= (H+32) else None

def add_calendar(pdf: pd.DataFrame, k=5) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    for wk in [47,48,49,50,51,52,1]:
        pdf[f"wk_{wk}"] = (woy==wk).astype(np.float32)
    return pdf

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred))/2.0
    m = denom != 0
    return float(100.0*np.mean(np.abs(y_pred[m]-y_true[m])/denom[m]))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((y_pred-y_true)**2)))

# ---- safety helpers (dtype & shape hardening) ----
def _coerce_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only numeric cols; coerce to float32."""
    if df is None or df.empty:
        return df
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        return pd.DataFrame(index=df.index)
    out = df[num_cols].apply(pd.to_numeric, errors="coerce").astype(np.float32)
    return out

def _coerce_cols_numeric_inplace(df: pd.DataFrame, cols: list[str]):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype(np.float32)

def _assert_all_float(df: pd.DataFrame, name: str):
    bad = [f"{c}:{str(df[c].dtype)}" for c in df.columns if not pd.api.types.is_float_dtype(df[c])]
    if bad:
        raise TypeError(f"{name} has non-float dtypes -> " + ", ".join(bad))

def _align_feature_frames(tr_p: pd.DataFrame, te_p: pd.DataFrame):
    """Make tr_p and te_p have identical columns (union), same order, float32."""
    tr_p = _coerce_numeric_df(tr_p).copy()
    te_p = _coerce_numeric_df(te_p).copy()
    cols = sorted(set(tr_p.columns).union(set(te_p.columns)))
    tr_p = tr_p.reindex(columns=cols, fill_value=0.0).astype(np.float32)
    te_p = te_p.reindex(columns=cols, fill_value=0.0).astype(np.float32)
    return tr_p, te_p

# ---- Model (load once)
try:
    module
except NameError:
    module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

def make_auto_feature_plan(n_total: int, h: int, ctx_ratio: float):
    n_train = n_total - h
    max_lag = int(max(8, min(52, 0.25*n_train)))  # allow yearly lag
    roll_candidates = [6,12,26,52]
    roll_means = [w for w in roll_candidates if w <= max(max_lag, 12)]
    roll_stds  = [w for w in [6,12,26] if w <= max(max_lag, 12)]
    ema_spans  = [s for s in [6,12] if s <= max_lag]
    warm = max(max_lag, max(roll_means+[6]), max(roll_stds+[6]), max(ema_spans+[6]))
    warm = int(min(warm, max(12, n_train//3)))
    ctx = n_train - warm
    ctx = max(CTX_FLOOR, min(CTX_CAP, int(ctx_ratio*ctx)))
    return {
        "lags": list(range(1, max_lag+1)),
        "roll_means": roll_means,
        "roll_stds": roll_stds,
        "ema_spans": ema_spans,
        "warm": warm,
        "ctx": ctx,
        "neigh_lags": list(range(1, min(6, max_lag)+1)),
        "exog_lags": list(range(1, min(6, max_lag)+1))
    }

def add_past_feats(pdf: pd.DataFrame, target: str, plan: dict) -> pd.DataFrame:
    pdf = pdf.copy()
    for L in plan["lags"]:
        pdf[f"{target}_lag_{L}"] = pdf[target].shift(L)
    for W in plan["roll_means"]:
        pdf[f"{target}_rollmean_{W}"] = pdf[target].rolling(W, min_periods=1).mean().shift(1)
    for W in plan["roll_stds"]:
        pdf[f"{target}_rollstd_{W}"] = pdf[target].rolling(W, min_periods=2).std().shift(1)
    for S in plan["ema_spans"]:
        pdf[f"{target}_ema_{S}"] = pdf[target].ewm(span=S, min_periods=1, adjust=False).mean().shift(1)
    return pdf

def _rolling_corr_spearman(a: pd.Series, b: pd.Series, w: int) -> float:
    a_w = pd.to_numeric(a.tail(w), errors="coerce")
    b_w = pd.to_numeric(b.tail(w), errors="coerce")
    if a_w.notna().sum() < 6 or b_w.notna().sum() < 6:
        return np.nan
    return a_w.corr(b_w, method="spearman")

def pick_top_neighbors(tr: pd.DataFrame, k=NEIGH_K):
    # only numeric neighbor candidates
    cand = [c for c in tr.columns if c.startswith(NEIGH_PREFIX) and pd.api.types.is_numeric_dtype(tr[c])]
    if not cand:
        return []
    y = pd.to_numeric(tr["y"], errors="coerce")
    scores = []
    for c in cand:
        s = pd.to_numeric(tr[c], errors="coerce")
        if s.isna().all():
            continue
        r = _rolling_corr_spearman(y, s, NEIGH_ROLL_W)
        if pd.isna(r):
            continue
        scores.append((abs(r), c))
    scores.sort(reverse=True)
    # mild diversity filter
    selected, selected_series = [], []
    for _, c in scores:
        s = pd.to_numeric(tr[c], errors="coerce")
        ok = True
        for s2 in selected_series:
            if abs(pd.Series(s).corr(pd.Series(s2))) > 0.95:
                ok = False; break
        if ok:
            selected.append(c); selected_series.append(s)
        if len(selected) >= k:
            break
    return selected

def list_past_exog_cols(df: pd.DataFrame):
    exog = []
    for c in df.columns:
        if c in ("ds","y","y_orig","y_cap"):
            continue
        if c.startswith(("cal_","wk_")) or c.startswith(NEIGH_PREFIX):
            continue
        if c in ("avg_sim_top25_last_week","n_sim_last_week"):
            exog.append(c)
        if c.startswith(("n_dev_","n_st_","impressions_sum","adclicks_sum","adcost_sum")):
            exog.append(c)
    # keep only numeric exog
    exog = [c for c in dict.fromkeys(exog) if pd.api.types.is_numeric_dtype(df[c])]
    return exog

def build_past_exog(tr: pd.DataFrame, te: pd.DataFrame, exog_cols: list, plan: dict):
    tr_ex = pd.DataFrame(index=tr.index)
    te_ex = pd.DataFrame(index=te.index)
    names = []
    if not exog_cols:
        return tr_ex, te_ex, names
    for c in exog_cols:
        Lmax = max(plan["exog_lags"])
        te_concat = pd.concat([tr[[c]].iloc[-Lmax:], te[[c]]], ignore_index=True)
        for L in plan["exog_lags"]:
            cname = f"{c}_lag{L}"
            tr_ex[cname] = pd.to_numeric(tr[c], errors="coerce").shift(L)
            te_ex[cname] = pd.to_numeric(te_concat[c], errors="coerce").shift(1).iloc[-len(te):].values
            names.append(cname)
    tr_ex = _coerce_numeric_df(tr_ex)
    te_ex = _coerce_numeric_df(te_ex)
    return tr_ex, te_ex, names

def robust_scale_train_test(tr: pd.DataFrame, te: pd.DataFrame, col: str):
    tr = tr.copy(); te = te.copy()
    m = tr[col].mean(); s = tr[col].std()
    if not np.isfinite(s) or s < 1e-6:
        med = tr[col].median()
        mad = (np.abs(tr[col]-med).median()) * 1.4826 + 1e-8
        tr[col] = (tr[col]-med)/mad
        te[col] = (te[col]-med)/mad
        return tr, te, ("robust", med, mad)
    else:
        tr[col] = (tr[col]-m)/(s+1e-8)
        te[col] = (te[col]-m)/(s+1e-8)
        return tr, te, ("standard", m, s+1e-8)

def _predict_moirai(module, plan, tr, te, tr_p, te_p, fut_cols):
    entry = {
        "start": pd.Timestamp(tr["ds"].iloc[0]),
        "target": tr["y"].to_numpy(np.float32),
    }
    if fut_cols:
        Xp = np.stack([tr[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        Xf = np.stack([te[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)

    past_cols = list(tr_p.columns)
    if past_cols:
        entry["past_feat_dynamic_real"] = np.stack([tr_p[c].to_numpy(np.float32) for c in past_cols], axis=0)

    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=len(fut_cols),
        past_feat_dynamic_real_dim=len(past_cols),
        context_length=plan["ctx"],
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    fc = next(predictor.predict(ListDataset([entry], freq=FREQ)))

    if fc.samples is not None:
        pred_stdlog = np.median(np.asarray(fc.samples, dtype=np.float32), axis=0)
    else:
        pred_stdlog = np.asarray(fc.mean, dtype=np.float32)
    return pred_stdlog

def _ridge_fit_predict(X_tr: pd.DataFrame, y_tr: pd.Series, X_te: pd.DataFrame, l2: float = 1.0):
    # Ensure identical columns/order between train/test for ridge
    X_tr = _coerce_numeric_df(X_tr)
    X_te = _coerce_numeric_df(X_te)
    if X_tr is None or X_tr.empty:
        return np.zeros(H, dtype=np.float32)
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0.0).astype(np.float32)

    X = np.asarray(X_tr, dtype=np.float32)
    y = np.asarray(y_tr, dtype=np.float32)
    XtX = X.T @ X
    XtX_reg = XtX + l2 * np.eye(XtX.shape[0], dtype=np.float32)
    Xty = X.T @ y
    try:
        w = np.linalg.solve(XtX_reg, Xty)
    except np.linalg.LinAlgError:
        w = np.linalg.pinv(XtX_reg) @ Xty
    pred = np.asarray(X_te, dtype=np.float32) @ w
    return pred

def _seasonal_naive_stdlog(tr_s: pd.Series, horizon: int, seasonal: int = 52):
    y = tr_s.reset_index(drop=True)
    n = len(y)
    out = np.empty(horizon, dtype=np.float32)
    for i in range(horizon):
        idx = n - seasonal + i
        if 0 <= idx < n:
            out[i] = y.iloc[idx]
        else:
            out[i] = y.iloc[-1]
    return out

def run_one(pdf: pd.DataFrame):
    pdf = add_calendar(pdf, k=5)

    # Winsorize extreme CPCs, keep original
    pdf = pdf.copy()
    pdf["y_orig"] = pdf["y"].astype(np.float32)
    cap = np.nanquantile(pdf["y_orig"].values, WINSOR_P) if np.isfinite(pdf["y_orig"]).any() else None
    pdf["y_cap"] = np.clip(pdf["y_orig"], 0, cap).astype(np.float32) if (cap is not None and np.isfinite(cap)) else np.clip(pdf["y_orig"], 0, None).astype(np.float32)

    # Log1p transform
    pdf["y"] = np.log1p(pdf["y_cap"]).astype(np.float32)

    tr, te = split_train_test(pdf, H)
    if tr is None:
        return np.nan, np.nan, 0, 0

    # Two plans for Moirai (ensemble)
    plan_short = make_auto_feature_plan(len(pdf), H, ctx_ratio=0.60)
    plan_long  = make_auto_feature_plan(len(pdf), H, ctx_ratio=0.90)

    # Scale on TRAIN ONLY (robust fallback)
    tr_s, te_s, scale_info = robust_scale_train_test(tr, te, "y")

    def build_for_plan(plan, tr_s, te_s):
        tr_p = add_past_feats(tr_s, "y", plan)
        te_build = pd.concat([tr_s.tail(plan["warm"]), te_s], ignore_index=True)
        te_p_all = add_past_feats(te_build, "y", plan)
        te_p     = te_p_all.iloc[plan["warm"]:]
        # Recency-aware neighbors
        neigh = pick_top_neighbors(tr_s, k=NEIGH_K)
        for c in neigh:
            for L in plan["neigh_lags"]:
                tr_p[f"{c}_lag{L}"] = pd.to_numeric(tr_s[c], errors="coerce").shift(L)
                te_concat = pd.concat([tr_s[[c]].iloc[-L:], te_s[[c]]], ignore_index=True)
                te_p[f"{c}_lag{L}"] = pd.to_numeric(te_concat[c], errors="coerce").shift(1).iloc[-len(te_s):].values
        # Past exogenous (lagged)
        exog_cols = list_past_exog_cols(pdf)
        tr_ex, te_ex, ex_names = build_past_exog(tr_s, te_s, exog_cols, plan)
        if not tr_ex.empty:
            tr_p = pd.concat([tr_p, tr_ex], axis=1)
            te_p = pd.concat([te_p, te_ex], axis=1)

        # DTYPE safety
        tr_p = _coerce_numeric_df(tr_p)
        te_p = _coerce_numeric_df(te_p)

        # SHAPE safety: align columns between train/test features
        tr_p, te_p = _align_feature_frames(tr_p, te_p)

        if len(tr_p) <= plan["warm"] + 8:
            return None

        tr_base = tr_s.iloc[plan["warm"]:].copy()
        te_base = te_s.copy()

        # Future-known features: calendar/week dummies -> ensure float32
        fut_cols = [c for c in pdf.columns if c.startswith(("cal_","wk_"))]
        _coerce_cols_numeric_inplace(tr_base, fut_cols)
        _coerce_cols_numeric_inplace(te_base, fut_cols)

        # Final assertion (dev aid)
        _assert_all_float(tr_p, "tr_p")
        _assert_all_float(te_p, "te_p")

        return dict(plan=plan, tr_base=tr_base, te_base=te_base, tr_p=tr_p, te_p=te_p, fut_cols=fut_cols)

    pack_s = build_for_plan(plan_short, tr_s, te_s)
    pack_l = build_for_plan(plan_long,  tr_s, te_s)
    if pack_s is None and pack_l is None:
        return np.nan, np.nan, 0, 0

    packs = [p for p in [pack_s, pack_l] if p is not None]

    # Moirai predictions (standardized log-space)
    preds_moirai = []
    for pack in packs:
        preds_moirai.append(
            _predict_moirai(module, pack["plan"], pack["tr_base"], pack["te_base"], pack["tr_p"], pack["te_p"], pack["fut_cols"])
        )
    pred_moirai_stdlog = np.mean(np.vstack(preds_moirai), axis=0) if len(preds_moirai) > 1 else preds_moirai[0]

    if PURE_MOIRAI:
        pred_stdlog_blend = pred_moirai_stdlog
    else:
        # Ridge on past features (use richer plan if available)
        pack_r = pack_l if pack_l is not None else pack_s
        pred_ridge_stdlog = _ridge_fit_predict(pack_r["tr_p"], pack_r["tr_base"]["y"], pack_r["te_p"], l2=0.5)
        # Seasonal naive (52w)
        pred_seasonal_stdlog = _seasonal_naive_stdlog(tr_s["y"], horizon=H, seasonal=52)
        # Blend
        pred_stdlog_blend = (
            W_MOIRAI   * pred_moirai_stdlog +
            W_RIDGE    * pred_ridge_stdlog +
            W_SEASONAL * pred_seasonal_stdlog
        )

    # Unscale to log space
    if scale_info[0] == "standard":
        _, m, s = scale_info
        pred_log = pred_stdlog_blend * s + m
    else:
        _, med, mad = scale_info
        pred_log = pred_stdlog_blend * mad + med

    # Back to original space
    yhat   = np.expm1(pred_log).astype(np.float32)
    y_true = te["y_cap"].to_numpy(np.float32)

    # Report feature counts from richer plan (long if available)
    any_pack = pack_l if pack_l is not None else pack_s
    kf = len(any_pack["fut_cols"]) if any_pack else 0
    kp = any_pack["tr_p"].shape[1] if any_pack else 0
    return smape(y_true, yhat), rmse(y_true, yhat), kf, kp

# ------------ Run (H=6 only) ------------
rows=[]
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} files.")
for p in tqdm(files, desc="H=6 auto-adaptive"):
    pdf = prep_keyword_pdf(p)
    if pdf is None:
        continue
    try:
        s, r, kf, kp = run_one(pdf)
    except Exception as e:
        print(f"[WARN] {p.stem} failed: {e}")
        s, r, kf, kp = np.nan, np.nan, 0, 0
    rows.append({
        "keyword": p.stem, "exog_setting": ("moirai-only" if PURE_MOIRAI else "moirai(short+long)+ridge+seasonal"),
        "horizon_weeks": H, "smape_canonical": s, "rmse": r,
        "n_future_covars": kf, "n_past_covars": kp, "n_total": len(pdf)
    })

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print("✅ Done. Metrics:", OUT_CSV)

valid = out.dropna(subset=["smape_canonical","rmse"])
if not valid.empty:
    g = valid.agg(
        count=("rmse","count"),
        sMAPE_mean=("smape_canonical","mean"),
        sMAPE_median=("smape_canonical","median"),
        RMSE_mean=("rmse","mean"),
        RMSE_median=("rmse","median"),
    )
    display(g)
    for t in [10,15,20,25,30,40]:
        share = (valid["smape_canonical"] <= t).mean()
        print(f"sMAPE≤{t:>2}: {share:.3f}")
else:
    print("No valid rows.")


In [ ]:
# ==== Moirai H=6 – IMPROVED v4.3 (pure Moirai default + recency anchoring + dtype/shape safety) ====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ------------ Fixed eval params ------------
FREQ        = "W"
H           = 6
NUM_SAMPLES = 320         # a bit more smoothing
BATCH_SIZE  = 64
CTX_CAP     = 160         # more context if VRAM allows
CTX_FLOOR   = 48
WINSOR_P    = 0.99        # slightly stronger cap than 0.995

NEIGH_PREFIX = "cpc_lastweek__"
NEIGH_K      = 6
NEIGH_ROLL_W = 52

# Blending settings
PURE_MOIRAI = True        # default: measure pure Moirai
W_MOIRAI   = 0.65
W_RIDGE    = 0.25
W_SEASONAL = 0.10

# Recency anchoring (leak-free): blend last observed level into forecasts
RECENCY_ANCHOR_ALPHA = 0.15  # 0.10–0.25 works well; set 0 to disable

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device)

# ------------ Paths ------------
IN_DIR  = KEYWORDS_DIR_20
OUT_DIR = moirai_results_dir("sem20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_H6_auto_adaptive.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# ------------ IO + helpers ------------
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")  # expects "WW-YYYY"
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns: return None
    base = (
        df_pl.select(pl.col("week"), pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week").sort("ds").filter(pl.col("y").is_not_null())
    ).to_pandas()
    base["ds"] = pd.to_datetime(base["ds"])
    base["y"]  = pd.to_numeric(base["y"], errors="coerce").astype(np.float32)
    base = base.sort_values("ds").reset_index(drop=True)
    return base if len(base) >= (H+32) else None

def add_calendar(pdf: pd.DataFrame, k=5) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    for wk in [47,48,49,50,51,52,1]:
        pdf[f"wk_{wk}"] = (woy==wk).astype(np.float32)
    return pdf

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred))/2.0
    m = denom != 0
    return float(100.0*np.mean(np.abs(y_pred[m]-y_true[m])/denom[m]))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((y_pred-y_true)**2)))

# ---- safety helpers (dtype & shape) ----
def _coerce_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        return pd.DataFrame(index=df.index)
    return df[num_cols].apply(pd.to_numeric, errors="coerce").astype(np.float32)

def _coerce_cols_numeric_inplace(df: pd.DataFrame, cols: list[str]):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype(np.float32)

def _assert_all_float(df: pd.DataFrame, name: str):
    bad = [f"{c}:{str(df[c].dtype)}" for c in df.columns if not pd.api.types.is_float_dtype(df[c])]
    if bad:
        raise TypeError(f"{name} has non-float dtypes -> " + ", ".join(bad))

def _align_feature_frames(tr_p: pd.DataFrame, te_p: pd.DataFrame):
    tr_p = _coerce_numeric_df(tr_p).copy()
    te_p = _coerce_numeric_df(te_p).copy()
    cols = sorted(set(tr_p.columns).union(set(te_p.columns)))
    tr_p = tr_p.reindex(columns=cols, fill_value=0.0).astype(np.float32)
    te_p = te_p.reindex(columns=cols, fill_value=0.0).astype(np.float32)
    return tr_p, te_p

# ---- Model (load once)
try:
    module
except NameError:
    module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

def make_auto_feature_plan(n_total: int, h: int, ctx_ratio: float):
    n_train = n_total - h
    max_lag = int(max(8, min(52, 0.25*n_train)))
    roll_candidates = [6,12,26,52]
    roll_means = [w for w in roll_candidates if w <= max(max_lag, 12)]
    roll_stds  = [w for w in [6,12,26] if w <= max(max_lag, 12)]
    ema_spans  = [s for s in [6,12] if s <= max_lag]
    warm = max(max_lag, max(roll_means+[6]), max(roll_stds+[6]), max(ema_spans+[6]))
    warm = int(min(warm, max(12, n_train//3)))
    ctx = n_train - warm
    ctx = max(CTX_FLOOR, min(CTX_CAP, int(ctx_ratio*ctx)))
    return {
        "lags": list(range(1, max_lag+1)),
        "roll_means": roll_means,
        "roll_stds": roll_stds,
        "ema_spans": ema_spans,
        "warm": warm,
        "ctx": ctx,
        "neigh_lags": list(range(1, min(6, max_lag)+1)),
        "exog_lags": list(range(1, min(6, max_lag)+1))
    }

def add_past_feats(pdf: pd.DataFrame, target: str, plan: dict) -> pd.DataFrame:
    pdf = pdf.copy()
    for L in plan["lags"]:
        pdf[f"{target}_lag_{L}"] = pdf[target].shift(L)
    for W in plan["roll_means"]:
        pdf[f"{target}_rollmean_{W}"] = pdf[target].rolling(W, min_periods=1).mean().shift(1)
    for W in plan["roll_stds"]:
        pdf[f"{target}_rollstd_{W}"] = pdf[target].rolling(W, min_periods=2).std().shift(1)
    for S in plan["ema_spans"]:
        pdf[f"{target}_ema_{S}"] = pdf[target].ewm(span=S, min_periods=1, adjust=False).mean().shift(1)
    return pdf

def _rolling_corr_spearman(a: pd.Series, b: pd.Series, w: int) -> float:
    a_w = pd.to_numeric(a.tail(w), errors="coerce")
    b_w = pd.to_numeric(b.tail(w), errors="coerce")
    if a_w.notna().sum() < 6 or b_w.notna().sum() < 6:
        return np.nan
    return a_w.corr(b_w, method="spearman")

def pick_top_neighbors(tr: pd.DataFrame, k=NEIGH_K):
    cand = [c for c in tr.columns if c.startswith(NEIGH_PREFIX) and pd.api.types.is_numeric_dtype(tr[c])]
    if not cand:
        return []
    y = pd.to_numeric(tr["y"], errors="coerce")
    scores = []
    for c in cand:
        s = pd.to_numeric(tr[c], errors="coerce")
        if s.isna().all():
            continue
        r = _rolling_corr_spearman(y, s, NEIGH_ROLL_W)
        if pd.isna(r):
            continue
        scores.append((abs(r), c))
    scores.sort(reverse=True)
    selected, selected_series = [], []
    for _, c in scores:
        s = pd.to_numeric(tr[c], errors="coerce")
        ok = True
        for s2 in selected_series:
            if abs(pd.Series(s).corr(pd.Series(s2))) > 0.95:
                ok = False; break
        if ok:
            selected.append(c); selected_series.append(s)
        if len(selected) >= k:
            break
    return selected

def list_past_exog_cols(df: pd.DataFrame):
    exog = []
    for c in df.columns:
        if c in ("ds","y","y_orig","y_cap"):
            continue
        if c.startswith(("cal_","wk_")) or c.startswith(NEIGH_PREFIX):
            continue
        if c in ("avg_sim_top25_last_week","n_sim_last_week"):
            exog.append(c)
        if c.startswith(("n_dev_","n_st_","impressions_sum","adclicks_sum","adcost_sum")):
            exog.append(c)
    exog = [c for c in dict.fromkeys(exog) if pd.api.types.is_numeric_dtype(df[c])]
    return exog

def build_past_exog(tr: pd.DataFrame, te: pd.DataFrame, exog_cols: list, plan: dict):
    tr_ex = pd.DataFrame(index=tr.index)
    te_ex = pd.DataFrame(index=te.index)
    if not exog_cols:
        return tr_ex, te_ex, []
    for c in exog_cols:
        Lmax = max(plan["exog_lags"])
        te_concat = pd.concat([tr[[c]].iloc[-Lmax:], te[[c]]], ignore_index=True)
        for L in plan["exog_lags"]:
            cname = f"{c}_lag{L}"
            tr_ex[cname] = pd.to_numeric(tr[c], errors="coerce").shift(L)
            te_ex[cname] = pd.to_numeric(te_concat[c], errors="coerce").shift(1).iloc[-len(te):].values
    tr_ex = _coerce_numeric_df(tr_ex)
    te_ex = _coerce_numeric_df(te_ex)
    return tr_ex, te_ex, list(tr_ex.columns)

def robust_scale_train_test(tr: pd.DataFrame, te: pd.DataFrame, col: str):
    tr = tr.copy(); te = te.copy()
    m = tr[col].mean(); s = tr[col].std()
    if not np.isfinite(s) or s < 1e-6:
        med = tr[col].median()
        mad = (np.abs(tr[col]-med).median()) * 1.4826 + 1e-8
        tr[col] = (tr[col]-med)/mad
        te[col] = (te[col]-med)/mad
        return tr, te, ("robust", med, mad)
    else:
        tr[col] = (tr[col]-m)/(s+1e-8)
        te[col] = (te[col]-m)/(s+1e-8)
        return tr, te, ("standard", m, s+1e-8)

def _predict_moirai(module, plan, tr, te, tr_p, te_p, fut_cols):
    entry = {
        "start": pd.Timestamp(tr["ds"].iloc[0]),
        "target": tr["y"].to_numpy(np.float32),
    }
    if fut_cols:
        Xp = np.stack([tr[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        Xf = np.stack([te[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)

    past_cols = list(tr_p.columns)
    if past_cols:
        entry["past_feat_dynamic_real"] = np.stack([tr_p[c].to_numpy(np.float32) for c in past_cols], axis=0)

    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=len(fut_cols),
        past_feat_dynamic_real_dim=len(past_cols),
        context_length=plan["ctx"],
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    fc = next(predictor.predict(ListDataset([entry], freq=FREQ)))

    if fc.samples is not None:
        pred_stdlog = np.median(np.asarray(fc.samples, dtype=np.float32), axis=0)
    else:
        pred_stdlog = np.asarray(fc.mean, dtype=np.float32)
    return pred_stdlog

def _ridge_fit_predict(X_tr: pd.DataFrame, y_tr: pd.Series, X_te: pd.DataFrame, l2: float = 1.0):
    X_tr = _coerce_numeric_df(pd.DataFrame(X_tr))
    X_te = _coerce_numeric_df(pd.DataFrame(X_te))
    y_tr = pd.to_numeric(pd.Series(y_tr), errors="coerce")
    if not X_tr.empty:
        keep = (~X_tr.isna().any(axis=1)) & y_tr.notna()
        X_tr = X_tr.loc[keep]; y_tr = y_tr.loc[keep]
    if X_tr.empty:
        return np.zeros(H, dtype=np.float32)
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0.0).astype(np.float32)
    n_features = X_tr.shape[1]
    if n_features == 0:
        return np.zeros(H, dtype=np.float32)
    X = np.asarray(X_tr, dtype=np.float32)
    y = np.asarray(y_tr, dtype=np.float32)
    try:
        XtX = X.T @ X
        XtX_reg = XtX + l2 * np.eye(n_features, dtype=np.float32)
        Xty = X.T @ y
        w = np.linalg.solve(XtX_reg, Xty)
        pred = np.asarray(X_te, dtype=np.float32) @ w
    except Exception:
        pred = np.zeros(H, dtype=np.float32)
    return pred

def _seasonal_naive_stdlog(tr_s: pd.Series, horizon: int, seasonal: int = 52):
    y = tr_s.reset_index(drop=True)
    n = len(y); out = np.empty(horizon, dtype=np.float32)
    for i in range(horizon):
        idx = n - seasonal + i
        out[i] = y.iloc[idx] if 0 <= idx < n else y.iloc[-1]
    return out

def _apply_recency_anchor(yhat: np.ndarray, last_level: float, alpha: float):
    """Blend last observed level into each horizon with linear decay."""
    if alpha <= 0:
        return yhat
    t = np.arange(1, len(yhat)+1, dtype=np.float32)
    # decays from alpha at t=1 down to ~alpha/2 at t=H (linear). Tune if desired.
    w = alpha * (1.0 - 0.5*(t-1)/max(1, len(yhat)-1))
    return (1.0 - w) * yhat + w * last_level

def run_one(pdf: pd.DataFrame):
    pdf = add_calendar(pdf, k=5)

    # Winsorize extreme CPCs, keep original
    pdf = pdf.copy()
    pdf["y_orig"] = pdf["y"].astype(np.float32)
    cap = np.nanquantile(pdf["y_orig"].values, WINSOR_P) if np.isfinite(pdf["y_orig"]).any() else None
    pdf["y_cap"] = np.clip(pdf["y_orig"], 0, cap).astype(np.float32) if (cap is not None and np.isfinite(cap)) else np.clip(pdf["y_orig"], 0, None).astype(np.float32)

    # Log1p transform
    pdf["y"] = np.log1p(pdf["y_cap"]).astype(np.float32)

    tr, te = split_train_test(pdf, H)
    if tr is None:
        return np.nan, np.nan, 0, 0

    # Two plans for Moirai (ensemble)
    plan_short = make_auto_feature_plan(len(pdf), H, ctx_ratio=0.60)
    plan_long  = make_auto_feature_plan(len(pdf), H, ctx_ratio=0.90)

    # Scale on TRAIN ONLY
    tr_s, te_s, scale_info = robust_scale_train_test(tr, te, "y")

    def build_for_plan(plan, tr_s, te_s):
        tr_p = add_past_feats(tr_s, "y", plan)
        te_build = pd.concat([tr_s.tail(plan["warm"]), te_s], ignore_index=True)
        te_p_all = add_past_feats(te_build, "y", plan)
        te_p     = te_p_all.iloc[plan["warm"]:]
        # Recency-aware neighbors
        neigh = pick_top_neighbors(tr_s, k=NEIGH_K)
        for c in neigh:
            for L in plan["neigh_lags"]:
                tr_p[f"{c}_lag{L}"] = pd.to_numeric(tr_s[c], errors="coerce").shift(L)
                te_concat = pd.concat([tr_s[[c]].iloc[-L:], te_s[[c]]], ignore_index=True)
                te_p[f"{c}_lag{L}"] = pd.to_numeric(te_concat[c], errors="coerce").shift(1).iloc[-len(te_s):].values
        # Past exogenous (lagged)
        exog_cols = list_past_exog_cols(pdf)
        tr_ex, te_ex, _ = build_past_exog(tr_s, te_s, exog_cols, plan)
        if not tr_ex.empty:
            tr_p = pd.concat([tr_p, tr_ex], axis=1)
            te_p = pd.concat([te_p, te_ex], axis=1)

        tr_p = _coerce_numeric_df(tr_p)
        te_p = _coerce_numeric_df(te_p)
        tr_p, te_p = _align_feature_frames(tr_p, te_p)

        if len(tr_p) <= plan["warm"] + 8:
            return None

        tr_base = tr_s.iloc[plan["warm"]:].copy()
        te_base = te_s.copy()
        fut_cols = [c for c in pdf.columns if c.startswith(("cal_","wk_"))]
        _coerce_cols_numeric_inplace(tr_base, fut_cols)
        _coerce_cols_numeric_inplace(te_base, fut_cols)
        _assert_all_float(tr_p, "tr_p"); _assert_all_float(te_p, "te_p")

        return dict(plan=plan, tr_base=tr_base, te_base=te_base, tr_p=tr_p, te_p=te_p, fut_cols=fut_cols)

    pack_s = build_for_plan(plan_short, tr_s, te_s)
    pack_l = build_for_plan(plan_long,  tr_s, te_s)
    if pack_s is None and pack_l is None:
        return np.nan, np.nan, 0, 0
    packs = [p for p in [pack_s, pack_l] if p is not None]

    # Moirai predictions (standardized log-space)
    preds_moirai = []
    for pack in packs:
        preds_moirai.append(
            _predict_moirai(module, pack["plan"], pack["tr_base"], pack["te_base"], pack["tr_p"], pack["te_p"], pack["fut_cols"])
        )
    pred_moirai_stdlog = np.mean(np.vstack(preds_moirai), axis=0) if len(preds_moirai) > 1 else preds_moirai[0]

    if PURE_MOIRAI:
        pred_stdlog_blend = pred_moirai_stdlog
    else:
        pack_r = pack_l if pack_l is not None else pack_s
        pred_ridge_stdlog = _ridge_fit_predict(pack_r["tr_p"], pack_r["tr_base"]["y"], pack_r["te_p"], l2=0.5)
        pred_seasonal_stdlog = _seasonal_naive_stdlog(tr_s["y"], horizon=H, seasonal=52)
        pred_stdlog_blend = (
            W_MOIRAI   * pred_moirai_stdlog +
            W_RIDGE    * pred_ridge_stdlog +
            W_SEASONAL * pred_seasonal_stdlog
        )

    # Unscale to log space
    if scale_info[0] == "standard":
        _, m, s = scale_info
        pred_log = pred_stdlog_blend * s + m
    else:
        _, med, mad = scale_info
        pred_log = pred_stdlog_blend * mad + med

    # Back to original space
    yhat   = np.expm1(pred_log).astype(np.float32)

    # Recency anchoring (leak-free): blend last observed level into each horizon
    last_level = float(tr["y_cap"].iloc[-1])
    if np.isfinite(last_level):
        yhat = _apply_recency_anchor(yhat, last_level, RECENCY_ANCHOR_ALPHA)

    y_true = te["y_cap"].to_numpy(np.float32)

    any_pack = pack_l if pack_l is not None else pack_s
    kf = len(any_pack["fut_cols"]) if any_pack else 0
    kp = any_pack["tr_p"].shape[1] if any_pack else 0
    return smape(y_true, yhat), rmse(y_true, yhat), kf, kp

# ------------ Run (H=6 only) ------------
rows=[]
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} files.")
for p in tqdm(files, desc="H=6 auto-adaptive"):
    pdf = prep_keyword_pdf(p)
    if pdf is None:
        continue
    try:
        s, r, kf, kp = run_one(pdf)
    except Exception as e:
        print(f"[WARN] {p.stem} failed: {e}")
        s, r, kf, kp = np.nan, np.nan, 0, 0
    rows.append({
        "keyword": p.stem, "exog_setting": ("moirai-only+anchor" if PURE_MOIRAI else "moirai+blend+anchor"),
        "horizon_weeks": H, "smape_canonical": s, "rmse": r,
        "n_future_covars": kf, "n_past_covars": kp, "n_total": len(pdf)
    })

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print("✅ Done. Metrics:", OUT_CSV)

valid = out.dropna(subset=["smape_canonical","rmse"])
if not valid.empty:
    g = valid.agg(
        count=("rmse","count"),
        sMAPE_mean=("smape_canonical","mean"),
        sMAPE_median=("smape_canonical","median"),
        RMSE_mean=("rmse","mean"),
        RMSE_median=("rmse","median"),
    )
    display(g)
    for t in [10,15,20,25,30,40]:
        share = (valid["smape_canonical"] <= t).mean()
        print(f"sMAPE≤{t:>2}: {share:.3f}")
else:
    print("No valid rows.")


In [ ]:
# ==== Moirai H=6 – IMPROVED v4.4 (pure Moirai + stronger calendar/recency + floor + dtype/shape safety) ====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ------------ Fixed eval params ------------
FREQ        = "W"
H           = 6
NUM_SAMPLES = 384         # more smoothing
BATCH_SIZE  = 64
CTX_CAP     = 192         # more context if VRAM allows
CTX_FLOOR   = 48
WINSOR_P    = 0.99        # stronger cap

NEIGH_PREFIX = "cpc_lastweek__"
NEIGH_K      = 8
NEIGH_ROLL_W = 78

# Mode (keep leakage-free)
PURE_MOIRAI = True        # set False to re-enable ridge/seasonal blend (kept but off)

# Recency anchoring (leak-free): blend last observed level into forecasts
RECENCY_ANCHOR_ALPHA = 0.20    # 0.10–0.25 typical
# Floor to avoid sMAPE explosions near zero (based ONLY on train)
EPS_FLOOR_PCT = 0.02           # 2% of train median(y_cap) as min prediction level

# Blend weights (only used if PURE_MOIRAI=False)
W_MOIRAI   = 0.65
W_RIDGE    = 0.25
W_SEASONAL = 0.10

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device)

# ------------ Paths ------------
IN_DIR  = KEYWORDS_DIR_20
OUT_DIR = moirai_results_dir("sem20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_H6_auto_adaptive.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# ------------ IO + helpers ------------
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")  # expects "WW-YYYY"
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns: return None
    base = (
        df_pl.select(pl.col("week"), pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week").sort("ds").filter(pl.col("y").is_not_null())
    ).to_pandas()
    base["ds"] = pd.to_datetime(base["ds"])
    base["y"]  = pd.to_numeric(base["y"], errors="coerce").astype(np.float32)
    base = base.sort_values("ds").reset_index(drop=True)
    return base if len(base) >= (H+32) else None

def add_calendar(pdf: pd.DataFrame, k=7) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    for wk in [47,48,49,50,51,52,1]:
        pdf[f"wk_{wk}"] = (woy==wk).astype(np.float32)
    return pdf

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred))/2.0
    m = denom != 0
    return float(100.0*np.mean(np.abs(y_pred[m]-y_true[m])/denom[m]))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((y_pred-y_true)**2)))

# ---- safety helpers (dtype & shape) ----
def _coerce_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        return pd.DataFrame(index=df.index)
    return df[num_cols].apply(pd.to_numeric, errors="coerce").astype(np.float32)

def _coerce_cols_numeric_inplace(df: pd.DataFrame, cols: list[str]):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype(np.float32)

def _assert_all_float(df: pd.DataFrame, name: str):
    bad = [f"{c}:{str(df[c].dtype)}" for c in df.columns if not pd.api.types.is_float_dtype(df[c])]
    if bad:
        raise TypeError(f"{name} has non-float dtypes -> " + ", ".join(bad))

def _align_feature_frames(tr_p: pd.DataFrame, te_p: pd.DataFrame):
    tr_p = _coerce_numeric_df(tr_p).copy()
    te_p = _coerce_numeric_df(te_p).copy()
    cols = sorted(set(tr_p.columns).union(set(te_p.columns)))
    tr_p = tr_p.reindex(columns=cols, fill_value=0.0).astype(np.float32)
    te_p = te_p.reindex(columns=cols, fill_value=0.0).astype(np.float32)
    return tr_p, te_p

# ---- Model (load once)
try:
    module
except NameError:
    module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

def make_auto_feature_plan(n_total: int, h: int, ctx_ratio: float):
    n_train = n_total - h
    max_lag = int(max(8, min(52, 0.25*n_train)))
    roll_candidates = [6,12,26,52]
    roll_means = [w for w in roll_candidates if w <= max(max_lag, 12)]
    roll_stds  = [w for w in [6,12,26] if w <= max(max_lag, 12)]
    ema_spans  = [s for s in [6,12] if s <= max_lag]
    warm = max(max_lag, max(roll_means+[6]), max(roll_stds+[6]), max(ema_spans+[6]))
    warm = int(min(warm, max(12, n_train//3)))
    ctx = n_train - warm
    ctx = max(CTX_FLOOR, min(CTX_CAP, int(ctx_ratio*ctx)))
    return {
        "lags": list(range(1, max_lag+1)),
        "roll_means": roll_means,
        "roll_stds": roll_stds,
        "ema_spans": ema_spans,
        "warm": warm,
        "ctx": ctx,
        "neigh_lags": list(range(1, min(6, max_lag)+1)),
        "exog_lags": list(range(1, min(6, max_lag)+1))
    }

def add_past_feats(pdf: pd.DataFrame, target: str, plan: dict) -> pd.DataFrame:
    pdf = pdf.copy()
    for L in plan["lags"]:
        pdf[f"{target}_lag_{L}"] = pdf[target].shift(L)
    for W in plan["roll_means"]:
        pdf[f"{target}_rollmean_{W}"] = pdf[target].rolling(W, min_periods=1).mean().shift(1)
    for W in plan["roll_stds"]:
        pdf[f"{target}_rollstd_{W}"] = pdf[target].rolling(W, min_periods=2).std().shift(1)
    for S in plan["ema_spans"]:
        pdf[f"{target}_ema_{S}"] = pdf[target].ewm(span=S, min_periods=1, adjust=False).mean().shift(1)
    return pdf

def _rolling_corr_spearman(a: pd.Series, b: pd.Series, w: int) -> float:
    a_w = pd.to_numeric(a.tail(w), errors="coerce")
    b_w = pd.to_numeric(b.tail(w), errors="coerce")
    if a_w.notna().sum() < 6 or b_w.notna().sum() < 6:
        return np.nan
    return a_w.corr(b_w, method="spearman")

def pick_top_neighbors(tr: pd.DataFrame, k=NEIGH_K):
    cand = [c for c in tr.columns if c.startswith(NEIGH_PREFIX) and pd.api.types.is_numeric_dtype(tr[c])]
    if not cand:
        return []
    y = pd.to_numeric(tr["y"], errors="coerce")
    scores = []
    for c in cand:
        s = pd.to_numeric(tr[c], errors="coerce")
        if s.isna().all():
            continue
        r = _rolling_corr_spearman(y, s, NEIGH_ROLL_W)
        if pd.isna(r):
            continue
        scores.append((abs(r), c))
    scores.sort(reverse=True)
    selected, selected_series = [], []
    for _, c in scores:
        s = pd.to_numeric(tr[c], errors="coerce")
        ok = True
        for s2 in selected_series:
            if abs(pd.Series(s).corr(pd.Series(s2))) > 0.95:
                ok = False; break
        if ok:
            selected.append(c); selected_series.append(s)
        if len(selected) >= k:
            break
    return selected

def list_past_exog_cols(df: pd.DataFrame):
    exog = []
    for c in df.columns:
        if c in ("ds","y","y_orig","y_cap"):
            continue
        if c.startswith(("cal_","wk_")) or c.startswith(NEIGH_PREFIX):
            continue
        if c in ("avg_sim_top25_last_week","n_sim_last_week"):
            exog.append(c)
        if c.startswith(("n_dev_","n_st_","impressions_sum","adclicks_sum","adcost_sum")):
            exog.append(c)
    exog = [c for c in dict.fromkeys(exog) if pd.api.types.is_numeric_dtype(df[c])]
    return exog

def build_past_exog(tr: pd.DataFrame, te: pd.DataFrame, exog_cols: list, plan: dict):
    tr_ex = pd.DataFrame(index=tr.index)
    te_ex = pd.DataFrame(index=te.index)
    if not exog_cols:
        return tr_ex, te_ex, []
    for c in exog_cols:
        Lmax = max(plan["exog_lags"])
        te_concat = pd.concat([tr[[c]].iloc[-Lmax:], te[[c]]], ignore_index=True)
        for L in plan["exog_lags"]:
            cname = f"{c}_lag{L}"
            tr_ex[cname] = pd.to_numeric(tr[c], errors="coerce").shift(L)
            te_ex[cname] = pd.to_numeric(te_concat[c], errors="coerce").shift(1).iloc[-len(te):].values
    tr_ex = _coerce_numeric_df(tr_ex)
    te_ex = _coerce_numeric_df(te_ex)
    return tr_ex, te_ex, list(tr_ex.columns)

def robust_scale_train_test(tr: pd.DataFrame, te: pd.DataFrame, col: str):
    tr = tr.copy(); te = te.copy()
    m = tr[col].mean(); s = tr[col].std()
    if not np.isfinite(s) or s < 1e-6:
        med = tr[col].median()
        mad = (np.abs(tr[col]-med).median()) * 1.4826 + 1e-8
        tr[col] = (tr[col]-med)/mad
        te[col] = (te[col]-med)/mad
        return tr, te, ("robust", med, mad)
    else:
        tr[col] = (tr[col]-m)/(s+1e-8)
        te[col] = (te[col]-m)/(s+1e-8)
        return tr, te, ("standard", m, s+1e-8)

def _predict_moirai(module, plan, tr, te, tr_p, te_p, fut_cols):
    entry = {
        "start": pd.Timestamp(tr["ds"].iloc[0]),
        "target": tr["y"].to_numpy(np.float32),
    }
    if fut_cols:
        Xp = np.stack([tr[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        Xf = np.stack([te[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)

    past_cols = list(tr_p.columns)
    if past_cols:
        entry["past_feat_dynamic_real"] = np.stack([tr_p[c].to_numpy(np.float32) for c in past_cols], axis=0)

    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=len(fut_cols),
        past_feat_dynamic_real_dim=len(past_cols),
        context_length=plan["ctx"],
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    fc = next(predictor.predict(ListDataset([entry], freq=FREQ)))

    if fc.samples is not None:
        pred_stdlog = np.median(np.asarray(fc.samples, dtype=np.float32), axis=0)
    else:
        pred_stdlog = np.asarray(fc.mean, dtype=np.float32)
    return pred_stdlog

# (Ridge + seasonal kept for optional blend, unchanged from v4.3)
def _ridge_fit_predict(X_tr: pd.DataFrame, y_tr: pd.Series, X_te: pd.DataFrame, l2: float = 1.0):
    X_tr = _coerce_numeric_df(pd.DataFrame(X_tr))
    X_te = _coerce_numeric_df(pd.DataFrame(X_te))
    y_tr = pd.to_numeric(pd.Series(y_tr), errors="coerce")
    if not X_tr.empty:
        keep = (~X_tr.isna().any(axis=1)) & y_tr.notna()
        X_tr = X_tr.loc[keep]; y_tr = y_tr.loc[keep]
    if X_tr.empty:
        return np.zeros(H, dtype=np.float32)
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0.0).astype(np.float32)
    n_features = X_tr.shape[1]
    if n_features == 0:
        return np.zeros(H, dtype=np.float32)
    X = np.asarray(X_tr, dtype=np.float32)
    y = np.asarray(y_tr, dtype=np.float32)
    try:
        XtX = X.T @ X
        XtX_reg = XtX + l2 * np.eye(n_features, dtype=np.float32)
        Xty = X.T @ y
        w = np.linalg.solve(XtX_reg, Xty)
        pred = np.asarray(X_te, dtype=np.float32) @ w
    except Exception:
        pred = np.zeros(H, dtype=np.float32)
    return pred

def _seasonal_naive_stdlog(tr_s: pd.Series, horizon: int, seasonal: int = 52):
    y = tr_s.reset_index(drop=True)
    n = len(y); out = np.empty(horizon, dtype=np.float32)
    for i in range(horizon):
        idx = n - seasonal + i
        out[i] = y.iloc[idx] if 0 <= idx < n else y.iloc[-1]
    return out

def _apply_recency_anchor(yhat: np.ndarray, last_level: float, alpha: float):
    if alpha <= 0:
        return yhat
    t = np.arange(1, len(yhat)+1, dtype=np.float32)
    w = alpha * (1.0 - 0.5*(t-1)/max(1, len(yhat)-1))  # linear decay
    return (1.0 - w) * yhat + w * last_level

def run_one(pdf: pd.DataFrame):
    pdf = add_calendar(pdf, k=7)

    # Winsorize extreme CPCs, keep original
    pdf = pdf.copy()
    pdf["y_orig"] = pdf["y"].astype(np.float32)
    cap = np.nanquantile(pdf["y_orig"].values, WINSOR_P) if np.isfinite(pdf["y_orig"]).any() else None
    pdf["y_cap"] = np.clip(pdf["y_orig"], 0, cap).astype(np.float32) if (cap is not None and np.isfinite(cap)) else np.clip(pdf["y_orig"], 0, None).astype(np.float32)

    # Log1p transform
    pdf["y"] = np.log1p(pdf["y_cap"]).astype(np.float32)

    tr, te = split_train_test(pdf, H)
    if tr is None:
        return np.nan, np.nan, 0, 0

    # Two plans for Moirai (ensemble)
    plan_short = make_auto_feature_plan(len(pdf), H, ctx_ratio=0.60)
    plan_long  = make_auto_feature_plan(len(pdf), H, ctx_ratio=0.90)

    # Scale on TRAIN ONLY
    tr_s, te_s, scale_info = robust_scale_train_test(tr, te, "y")

    def build_for_plan(plan, tr_s, te_s):
        tr_p = add_past_feats(tr_s, "y", plan)
        te_build = pd.concat([tr_s.tail(plan["warm"]), te_s], ignore_index=True)
        te_p_all = add_past_feats(te_build, "y", plan)
        te_p     = te_p_all.iloc[plan["warm"]:]
        # Recency-aware neighbors
        neigh = pick_top_neighbors(tr_s, k=NEIGH_K)
        for c in neigh:
            for L in plan["neigh_lags"]:
                tr_p[f"{c}_lag{L}"] = pd.to_numeric(tr_s[c], errors="coerce").shift(L)
                te_concat = pd.concat([tr_s[[c]].iloc[-L:], te_s[[c]]], ignore_index=True)
                te_p[f"{c}_lag{L}"] = pd.to_numeric(te_concat[c], errors="coerce").shift(1).iloc[-len(te_s):].values
        # Past exogenous (lagged)
        exog_cols = list_past_exog_cols(pdf)
        tr_ex, te_ex, _ = build_past_exog(tr_s, te_s, exog_cols, plan)
        if not tr_ex.empty:
            tr_p = pd.concat([tr_p, tr_ex], axis=1)
            te_p = pd.concat([te_p, te_ex], axis=1)

        tr_p = _coerce_numeric_df(tr_p)
        te_p = _coerce_numeric_df(te_p)
        tr_p, te_p = _align_feature_frames(tr_p, te_p)

        if len(tr_p) <= plan["warm"] + 8:
            return None

        tr_base = tr_s.iloc[plan["warm"]:].copy()
        te_base = te_s.copy()
        fut_cols = [c for c in pdf.columns if c.startswith(("cal_","wk_"))]
        _coerce_cols_numeric_inplace(tr_base, fut_cols)
        _coerce_cols_numeric_inplace(te_base, fut_cols)
        _assert_all_float(tr_p, "tr_p"); _assert_all_float(te_p, "te_p")

        return dict(plan=plan, tr_base=tr_base, te_base=te_base, tr_p=tr_p, te_p=te_p, fut_cols=fut_cols)

    pack_s = build_for_plan(plan_short, tr_s, te_s)
    pack_l = build_for_plan(plan_long,  tr_s, te_s)
    if pack_s is None and pack_l is None:
        return np.nan, np.nan, 0, 0
    packs = [p for p in [pack_s, pack_l] if p is not None]

    # Moirai predictions (standardized log-space)
    preds_moirai = []
    for pack in packs:
        preds_moirai.append(
            _predict_moirai(module, pack["plan"], pack["tr_base"], pack["te_base"], pack["tr_p"], pack["te_p"], pack["fut_cols"])
        )
    pred_moirai_stdlog = np.mean(np.vstack(preds_moirai), axis=0) if len(preds_moirai) > 1 else preds_moirai[0]

    if PURE_MOIRAI:
        pred_stdlog_blend = pred_moirai_stdlog
    else:
        pack_r = pack_l if pack_l is not None else pack_s
        pred_ridge_stdlog = _ridge_fit_predict(pack_r["tr_p"], pack_r["tr_base"]["y"], pack_r["te_p"], l2=0.5)
        pred_seasonal_stdlog = _seasonal_naive_stdlog(tr_s["y"], horizon=H, seasonal=52)
        pred_stdlog_blend = (
            W_MOIRAI   * pred_moirai_stdlog +
            W_RIDGE    * pred_ridge_stdlog +
            W_SEASONAL * pred_seasonal_stdlog
        )

    # Unscale to log space
    if scale_info[0] == "standard":
        _, m, s = scale_info
        pred_log = pred_stdlog_blend * s + m
    else:
        _, med, mad = scale_info
        pred_log = pred_stdlog_blend * mad + med

    # Back to original space
    yhat = np.expm1(pred_log).astype(np.float32)

    # Recency anchoring (leak-free) + min floor (based on TRAIN ONLY)
    last_level = float(tr["y_cap"].iloc[-1])
    if np.isfinite(last_level):
        yhat = _apply_recency_anchor(yhat, last_level, RECENCY_ANCHOR_ALPHA)

    # Floor predictions using train median to reduce sMAPE blowups near zero
    train_median = float(np.nanmedian(tr["y_cap"].values))
    eps = float(max(1e-8, EPS_FLOOR_PCT * train_median)) if np.isfinite(train_median) else 1e-8
    yhat = np.clip(yhat, eps, None)

    y_true = te["y_cap"].to_numpy(np.float32)

    any_pack = pack_l if pack_l is not None else pack_s
    kf = len(any_pack["fut_cols"]) if any_pack else 0
    kp = any_pack["tr_p"].shape[1] if any_pack else 0
    return smape(y_true, yhat), rmse(y_true, yhat), kf, kp

# ------------ Run (H=6 only) ------------
rows=[]
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} files.")
for p in tqdm(files, desc="H=6 auto-adaptive"):
    pdf = prep_keyword_pdf(p)
    if pdf is None:
        continue
    try:
        s, r, kf, kp = run_one(pdf)
    except Exception as e:
        print(f"[WARN] {p.stem} failed: {e}")
        s, r, kf, kp = np.nan, np.nan, 0, 0
    rows.append({
        "keyword": p.stem, "exog_setting": ("moirai-only+anchor+floor" if PURE_MOIRAI else "moirai+blend+anchor+floor"),
        "horizon_weeks": H, "smape_canonical": s, "rmse": r,
        "n_future_covars": kf, "n_past_covars": kp, "n_total": len(pdf)
    })

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print("✅ Done. Metrics:", OUT_CSV)

valid = out.dropna(subset=["smape_canonical","rmse"])
if not valid.empty:
    g = valid.agg(
        count=("rmse","count"),
        sMAPE_mean=("smape_canonical","mean"),
        sMAPE_median=("smape_canonical","median"),
        RMSE_mean=("rmse","mean"),
        RMSE_median=("rmse","median"),
    )
    display(g)
    for t in [10,15,20,25,30,40]:
        share = (valid["smape_canonical"] <= t).mean()
        print(f"sMAPE≤{t:>2}: {share:.3f}")
else:
    print("No valid rows.")


### ↓ Canonical cell for the paper (see banner at the top of the notebook)

Below is the cell whose output is reported in the paper. All earlier cells in this notebook run, pin packages, or iterate the methodology that leads up to this cell.


In [ ]:
# ==== Moirai H=6 – PROD v4.5 (pure Moirai + k=9 calendar + α=0.25 anchor + 3% floor + dtype/shape safety) ====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ------------ Fixed eval params ------------
FREQ        = "W"
H           = 6
NUM_SAMPLES = 448         # more smoothing
BATCH_SIZE  = 64
CTX_CAP     = 224         # more context if VRAM allows
CTX_FLOOR   = 48
WINSOR_P    = 0.99        # cap extreme CPCs

NEIGH_PREFIX = "cpc_lastweek__"
NEIGH_K      = 8
NEIGH_ROLL_W = 78

# Mode (keep leakage-free)
PURE_MOIRAI = True

# Recency anchoring (leak-free): blend last observed level into forecasts
RECENCY_ANCHOR_ALPHA = 0.25    # stronger level stabilization
# Floor to avoid sMAPE explosions near zero (based ONLY on train)
EPS_FLOOR_PCT = 0.03           # 3% of train median(y_cap) as min prediction

# (Unused unless you flip PURE_MOIRAI to False)
W_MOIRAI   = 0.65
W_RIDGE    = 0.25
W_SEASONAL = 0.10

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device)

# ------------ Paths ------------
IN_DIR  = KEYWORDS_DIR_20
OUT_DIR = moirai_results_dir("sem20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_H6_auto_adaptive.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# ------------ IO + helpers ------------
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")  # expects "WW-YYYY"
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns: return None
    base = (
        df_pl.select(pl.col("week"), pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week").sort("ds").filter(pl.col("y").is_not_null())
    ).to_pandas()
    base["ds"] = pd.to_datetime(base["ds"])
    base["y"]  = pd.to_numeric(base["y"], errors="coerce").astype(np.float32)
    base = base.sort_values("ds").reset_index(drop=True)
    return base if len(base) >= (H+32) else None

def add_calendar(pdf: pd.DataFrame, k=9) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    for wk in [47,48,49,50,51,52,1]:
        pdf[f"wk_{wk}"] = (woy==wk).astype(np.float32)
    return pdf

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred))/2.0
    m = denom != 0
    return float(100.0*np.mean(np.abs(y_pred[m]-y_true[m])/denom[m]))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((y_pred-y_true)**2)))

# ---- safety helpers (dtype & shape) ----
def _coerce_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        return pd.DataFrame(index=df.index)
    return df[num_cols].apply(pd.to_numeric, errors="coerce").astype(np.float32)

def _coerce_cols_numeric_inplace(df: pd.DataFrame, cols: list[str]):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype(np.float32)

def _assert_all_float(df: pd.DataFrame, name: str):
    bad = [f"{c}:{str(df[c].dtype)}" for c in df.columns if not pd.api.types.is_float_dtype(df[c])]
    if bad:
        raise TypeError(f"{name} has non-float dtypes -> " + ", ".join(bad))

def _align_feature_frames(tr_p: pd.DataFrame, te_p: pd.DataFrame):
    tr_p = _coerce_numeric_df(tr_p).copy()
    te_p = _coerce_numeric_df(te_p).copy()
    cols = sorted(set(tr_p.columns).union(set(te_p.columns)))
    tr_p = tr_p.reindex(columns=cols, fill_value=0.0).astype(np.float32)
    te_p = te_p.reindex(columns=cols, fill_value=0.0).astype(np.float32)
    return tr_p, te_p

# ---- Model (load once)
try:
    module
except NameError:
    module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

def make_auto_feature_plan(n_total: int, h: int, ctx_ratio: float):
    n_train = n_total - h
    max_lag = int(max(8, min(52, 0.25*n_train)))
    roll_candidates = [6,12,26,52]
    roll_means = [w for w in roll_candidates if w <= max(max_lag, 12)]
    roll_stds  = [w for w in [6,12,26] if w <= max(max_lag, 12)]
    ema_spans  = [s for s in [6,12] if s <= max_lag]
    warm = max(max_lag, max(roll_means+[6]), max(roll_stds+[6]), max(ema_spans+[6]))
    warm = int(min(warm, max(12, n_train//3)))
    ctx = n_train - warm
    ctx = max(CTX_FLOOR, min(CTX_CAP, int(ctx_ratio*ctx)))
    return {
        "lags": list(range(1, max_lag+1)),
        "roll_means": roll_means,
        "roll_stds": roll_stds,
        "ema_spans": ema_spans,
        "warm": warm,
        "ctx": ctx,
        "neigh_lags": list(range(1, min(6, max_lag)+1)),
        "exog_lags": list(range(1, min(6, max_lag)+1))
    }

def add_past_feats(pdf: pd.DataFrame, target: str, plan: dict) -> pd.DataFrame:
    pdf = pdf.copy()
    for L in plan["lags"]:
        pdf[f"{target}_lag_{L}"] = pdf[target].shift(L)
    for W in plan["roll_means"]:
        pdf[f"{target}_rollmean_{W}"] = pdf[target].rolling(W, min_periods=1).mean().shift(1)
    for W in plan["roll_stds"]:
        pdf[f"{target}_rollstd_{W}"] = pdf[target].rolling(W, min_periods=2).std().shift(1)
    for S in plan["ema_spans"]:
        pdf[f"{target}_ema_{S}"] = pdf[target].ewm(span=S, min_periods=1, adjust=False).mean().shift(1)
    return pdf

def _rolling_corr_spearman(a: pd.Series, b: pd.Series, w: int) -> float:
    a_w = pd.to_numeric(a.tail(w), errors="coerce")
    b_w = pd.to_numeric(b.tail(w), errors="coerce")
    if a_w.notna().sum() < 6 or b_w.notna().sum() < 6:
        return np.nan
    return a_w.corr(b_w, method="spearman")

def pick_top_neighbors(tr: pd.DataFrame, k=NEIGH_K):
    cand = [c for c in tr.columns if c.startswith(NEIGH_PREFIX) and pd.api.types.is_numeric_dtype(tr[c])]
    if not cand:
        return []
    y = pd.to_numeric(tr["y"], errors="coerce")
    scores = []
    for c in cand:
        s = pd.to_numeric(tr[c], errors="coerce")
        if s.isna().all():
            continue
        r = _rolling_corr_spearman(y, s, NEIGH_ROLL_W)
        if pd.isna(r):
            continue
        scores.append((abs(r), c))
    scores.sort(reverse=True)
    selected, selected_series = [], []
    for _, c in scores:
        s = pd.to_numeric(tr[c], errors="coerce")
        ok = True
        for s2 in selected_series:
            if abs(pd.Series(s).corr(pd.Series(s2))) > 0.95:
                ok = False; break
        if ok:
            selected.append(c); selected_series.append(s)
        if len(selected) >= k:
            break
    return selected

def list_past_exog_cols(df: pd.DataFrame):
    exog = []
    for c in df.columns:
        if c in ("ds","y","y_orig","y_cap"):
            continue
        if c.startswith(("cal_","wk_")) or c.startswith(NEIGH_PREFIX):
            continue
        if c in ("avg_sim_top25_last_week","n_sim_last_week"):
            exog.append(c)
        if c.startswith(("n_dev_","n_st_","impressions_sum","adclicks_sum","adcost_sum")):
            exog.append(c)
    exog = [c for c in dict.fromkeys(exog) if pd.api.types.is_numeric_dtype(df[c])]
    return exog

def build_past_exog(tr: pd.DataFrame, te: pd.DataFrame, exog_cols: list, plan: dict):
    tr_ex = pd.DataFrame(index=tr.index); te_ex = pd.DataFrame(index=te.index)
    if not exog_cols:
        return tr_ex, te_ex, []
    for c in exog_cols:
        Lmax = max(plan["exog_lags"])
        te_concat = pd.concat([tr[[c]].iloc[-Lmax:], te[[c]]], ignore_index=True)
        for L in plan["exog_lags"]:
            cname = f"{c}_lag{L}"
            tr_ex[cname] = pd.to_numeric(tr[c], errors="coerce").shift(L)
            te_ex[cname] = pd.to_numeric(te_concat[c], errors="coerce").shift(1).iloc[-len(te):].values
    tr_ex = _coerce_numeric_df(tr_ex); te_ex = _coerce_numeric_df(te_ex)
    return tr_ex, te_ex, list(tr_ex.columns)

def robust_scale_train_test(tr: pd.DataFrame, te: pd.DataFrame, col: str):
    tr = tr.copy(); te = te.copy()
    m = tr[col].mean(); s = tr[col].std()
    if not np.isfinite(s) or s < 1e-6:
        med = tr[col].median()
        mad = (np.abs(tr[col]-med).median()) * 1.4826 + 1e-8
        tr[col] = (tr[col]-med)/mad
        te[col] = (te[col]-med)/mad
        return tr, te, ("robust", med, mad)
    else:
        tr[col] = (tr[col]-m)/(s+1e-8)
        te[col] = (te[col]-m)/(s+1e-8)
        return tr, te, ("standard", m, s+1e-8)

def _predict_moirai(module, plan, tr, te, tr_p, te_p, fut_cols):
    entry = {
        "start": pd.Timestamp(tr["ds"].iloc[0]),
        "target": tr["y"].to_numpy(np.float32),
    }
    if fut_cols:
        Xp = np.stack([tr[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        Xf = np.stack([te[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)

    past_cols = list(tr_p.columns)
    if past_cols:
        entry["past_feat_dynamic_real"] = np.stack([tr_p[c].to_numpy(np.float32) for c in past_cols], axis=0)

    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=len(fut_cols),
        past_feat_dynamic_real_dim=len(past_cols),
        context_length=plan["ctx"],
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    fc = next(predictor.predict(ListDataset([entry], freq=FREQ)))

    if fc.samples is not None:
        pred_stdlog = np.median(np.asarray(fc.samples, dtype=np.float32), axis=0)
    else:
        pred_stdlog = np.asarray(fc.mean, dtype=np.float32)
    return pred_stdlog

# (Ridge + seasonal kept for optional blend, but PURE_MOIRAI=True by default)
def _ridge_fit_predict(X_tr: pd.DataFrame, y_tr: pd.Series, X_te: pd.DataFrame, l2: float = 1.0):
    X_tr = _coerce_numeric_df(pd.DataFrame(X_tr))
    X_te = _coerce_numeric_df(pd.DataFrame(X_te))
    y_tr = pd.to_numeric(pd.Series(y_tr), errors="coerce")
    if not X_tr.empty:
        keep = (~X_tr.isna().any(axis=1)) & y_tr.notna()
        X_tr = X_tr.loc[keep]; y_tr = y_tr.loc[keep]
    if X_tr.empty:
        return np.zeros(H, dtype=np.float32)
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0.0).astype(np.float32)
    n_features = X_tr.shape[1]
    if n_features == 0:
        return np.zeros(H, dtype=np.float32)
    X = np.asarray(X_tr, dtype=np.float32); y = np.asarray(y_tr, dtype=np.float32)
    try:
        XtX = X.T @ X
        XtX_reg = XtX + l2 * np.eye(n_features, dtype=np.float32)
        Xty = X.T @ y
        w = np.linalg.solve(XtX_reg, Xty)
        pred = np.asarray(X_te, dtype=np.float32) @ w
    except Exception:
        pred = np.zeros(H, dtype=np.float32)
    return pred

def _seasonal_naive_stdlog(tr_s: pd.Series, horizon: int, seasonal: int = 52):
    y = tr_s.reset_index(drop=True); n = len(y)
    out = np.empty(horizon, dtype=np.float32)
    for i in range(horizon):
        idx = n - seasonal + i
        out[i] = y.iloc[idx] if 0 <= idx < n else y.iloc[-1]
    return out

def _apply_recency_anchor(yhat: np.ndarray, last_level: float, alpha: float):
    if alpha <= 0:
        return yhat
    t = np.arange(1, len(yhat)+1, dtype=np.float32)
    w = alpha * (1.0 - 0.5*(t-1)/max(1, len(yhat)-1))  # linear decay
    return (1.0 - w) * yhat + w * last_level

def run_one(pdf: pd.DataFrame):
    pdf = add_calendar(pdf, k=9)

    # Winsorize extreme CPCs, keep original
    pdf = pdf.copy()
    pdf["y_orig"] = pdf["y"].astype(np.float32)
    cap = np.nanquantile(pdf["y_orig"].values, WINSOR_P) if np.isfinite(pdf["y_orig"]).any() else None
    pdf["y_cap"] = np.clip(pdf["y_orig"], 0, cap).astype(np.float32) if (cap is not None and np.isfinite(cap)) else np.clip(pdf["y_orig"], 0, None).astype(np.float32)

    # Log1p transform
    pdf["y"] = np.log1p(pdf["y_cap"]).astype(np.float32)

    tr, te = split_train_test(pdf, H)
    if tr is None:
        return np.nan, np.nan, 0, 0

    # Two plans for Moirai (ensemble)
    plan_short = make_auto_feature_plan(len(pdf), H, ctx_ratio=0.60)
    plan_long  = make_auto_feature_plan(len(pdf), H, ctx_ratio=0.90)

    # Scale on TRAIN ONLY
    tr_s, te_s, scale_info = robust_scale_train_test(tr, te, "y")

    def build_for_plan(plan, tr_s, te_s):
        tr_p = add_past_feats(tr_s, "y", plan)
        te_build = pd.concat([tr_s.tail(plan["warm"]), te_s], ignore_index=True)
        te_p_all = add_past_feats(te_build, "y", plan)
        te_p     = te_p_all.iloc[plan["warm"]:]
        # Recency-aware neighbors
        neigh = pick_top_neighbors(tr_s, k=NEIGH_K)
        for c in neigh:
            for L in plan["neigh_lags"]:
                tr_p[f"{c}_lag{L}"] = pd.to_numeric(tr_s[c], errors="coerce").shift(L)
                te_concat = pd.concat([tr_s[[c]].iloc[-L:], te_s[[c]]], ignore_index=True)
                te_p[f"{c}_lag{L}"] = pd.to_numeric(te_concat[c], errors="coerce").shift(1).iloc[-len(te_s):].values
        # Past exogenous (lagged)
        exog_cols = list_past_exog_cols(pdf)
        tr_ex, te_ex, _ = build_past_exog(tr_s, te_s, exog_cols, plan)
        if not tr_ex.empty:
            tr_p = pd.concat([tr_p, tr_ex], axis=1)
            te_p = pd.concat([te_p, te_ex], axis=1)

        tr_p = _coerce_numeric_df(tr_p)
        te_p = _coerce_numeric_df(te_p)
        tr_p, te_p = _align_feature_frames(tr_p, te_p)

        if len(tr_p) <= plan["warm"] + 8:
            return None

        tr_base = tr_s.iloc[plan["warm"]:].copy()
        te_base = te_s.copy()
        fut_cols = [c for c in pdf.columns if c.startswith(("cal_","wk_"))]
        _coerce_cols_numeric_inplace(tr_base, fut_cols)
        _coerce_cols_numeric_inplace(te_base, fut_cols)
        _assert_all_float(tr_p, "tr_p"); _assert_all_float(te_p, "te_p")

        return dict(plan=plan, tr_base=tr_base, te_base=te_base, tr_p=tr_p, te_p=te_p, fut_cols=fut_cols)

    pack_s = build_for_plan(plan_short, tr_s, te_s)
    pack_l = build_for_plan(plan_long,  tr_s, te_s)
    if pack_s is None and pack_l is None:
        return np.nan, np.nan, 0, 0
    packs = [p for p in [pack_s, pack_l] if p is not None]

    # Moirai predictions (standardized log-space)
    preds_moirai = []
    for pack in packs:
        preds_moirai.append(
            _predict_moirai(module, pack["plan"], pack["tr_base"], pack["te_base"], pack["tr_p"], pack["te_p"], pack["fut_cols"])
        )
    pred_moirai_stdlog = np.mean(np.vstack(preds_moirai), axis=0) if len(preds_moirai) > 1 else preds_moirai[0]

    if PURE_MOIRAI:
        pred_stdlog_blend = pred_moirai_stdlog
    else:
        pack_r = pack_l if pack_l is not None else pack_s
        pred_ridge_stdlog = _ridge_fit_predict(pack_r["tr_p"], pack_r["tr_base"]["y"], pack_r["te_p"], l2=0.5)
        pred_seasonal_stdlog = _seasonal_naive_stdlog(tr_s["y"], horizon=H, seasonal=52)
        pred_stdlog_blend = (
            W_MOIRAI   * pred_moirai_stdlog +
            W_RIDGE    * pred_ridge_stdlog +
            W_SEASONAL * pred_seasonal_stdlog
        )

    # Unscale to log space
    if scale_info[0] == "standard":
        _, m, s = scale_info
        pred_log = pred_stdlog_blend * s + m
    else:
        _, med, mad = scale_info
        pred_log = pred_stdlog_blend * mad + med

    # Back to original space
    yhat = np.expm1(pred_log).astype(np.float32)

    # Recency anchoring (leak-free) + min floor (based on TRAIN ONLY)
    last_level = float(tr["y_cap"].iloc[-1])
    if np.isfinite(last_level):
        yhat = _apply_recency_anchor(yhat, last_level, RECENCY_ANCHOR_ALPHA)

    train_median = float(np.nanmedian(tr["y_cap"].values))
    eps = float(max(1e-8, EPS_FLOOR_PCT * train_median)) if np.isfinite(train_median) else 1e-8
    yhat = np.clip(yhat, eps, None)

    y_true = te["y_cap"].to_numpy(np.float32)

    any_pack = pack_l if pack_l is not None else pack_s
    kf = len(any_pack["fut_cols"]) if any_pack else 0
    kp = any_pack["tr_p"].shape[1] if any_pack else 0
    return smape(y_true, yhat), rmse(y_true, yhat), kf, kp

# ------------ Run (H=6 only) ------------
rows=[]
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} files.")
for p in tqdm(files, desc="H=6 auto-adaptive"):
    pdf = prep_keyword_pdf(p)
    if pdf is None:
        continue
    try:
        s, r, kf, kp = run_one(pdf)
    except Exception as e:
        print(f"[WARN] {p.stem} failed: {e}")
        s, r, kf, kp = np.nan, np.nan, 0, 0
    rows.append({
        "keyword": p.stem, "exog_setting": ("moirai-only+anchor+floor" if PURE_MOIRAI else "moirai+blend+anchor+floor"),
        "horizon_weeks": H, "smape_canonical": s, "rmse": r,
        "n_future_covars": kf, "n_past_covars": kp, "n_total": len(pdf)
    })

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print("✅ Done. Metrics:", OUT_CSV)

valid = out.dropna(subset=["smape_canonical","rmse"])
if not valid.empty:
    g = valid.agg(
        count=("rmse","count"),
        sMAPE_mean=("smape_canonical","mean"),
        sMAPE_median=("smape_canonical","median"),
        RMSE_mean=("rmse","mean"),
        RMSE_median=("rmse","median"),
    )
    display(g)
    for t in [10,15,20,25,30,40]:
        share = (valid["smape_canonical"] <= t).mean()
        print(f"sMAPE≤{t:>2}: {share:.3f}")
else:
    print("No valid rows.")
